# P2 – Finanças do Setor Público (UnB) – Reprodução completa
**Matrícula 232045821 · Hungria · Barreiras/BA e Camaçari/BA**

Rode tudo em ordem (*Ambiente de execução → Executar tudo*, ~10 min). Etapas: coleta (Siconfi, IBGE, Banco Mundial, Eurostat) → tratamento → base da P1 e auditoria → registro de alterações → análise (tabelas LaTeX e gráficos) → dicionário → download de `P2_completo.zip`. Nenhum valor ausente é trocado por zero.

In [ ]:
import os
os.makedirs('/content/P2/codigo', exist_ok=True)
%cd /content/P2
!pip -q install requests pandas numpy scipy matplotlib openpyxl

## Scripts do projeto
Estas células só gravam os arquivos. Para trocar municípios, país ou anos, edite apenas `config.py`.

In [ ]:
%%writefile codigo/config.py
"""
Configuração única do projeto P2. Para trocar de caso (municípios/país), edite SÓ este arquivo.

Municípios (código IBGE de 7 dígitos). O script de coleta VALIDA o nome pelo IBGE antes de baixar;
se algum código estiver errado, ele para e avisa.
"""
MUNICIPIOS = {
    2903201: "Barreiras",   # BA
    2905701: "Camaçari",    # BA
}
UF_ESPERADA = {2903201: "BA", 2905701: "BA"}

# País do caso atribuído (ISO3) e referência (Brasil)
PAIS_ISO3 = "HUN"
PAIS_NOME = "Hungria"
REFERENCIA_ISO3 = "BRA"
EUROSTAT_GEO = "HU"            # para países da UE; deixe None se não houver dados Eurostat
USAR_EUROSTAT = True

# Período e deflator
ANO_INI, ANO_FIM = 2014, 2025   # coleta 2014-2025: sobrepõe a P1 (2014-2023) para o registro de alterações.
JANELA_ANALISE = (2016, 2025)   # 10 anos mais recentes com DCA completa (conferir 2025 no log). PIB municipal só até 2023.
ANO_BASE = 2025                 # preços constantes de 2025 (IPCA média anual)

# População do Censo 2022 (usada onde a tabela 6579 do IBGE não traz 2022).
# O coleta_municipios.py baixa da tabela 4714 do SIDRA. Se falhar, preencha data/manual/censo2022.csv
# (colunas: cod_ibge,populacao) com o valor oficial do IBGE.


In [ ]:
%%writefile codigo/rubricas_receita.py
"""
Extração de rubricas de receita robusta às mudanças do ementário (2014-2017, 2018-2021, 2022+).

Problema: o código da mesma receita muda ao longo do tempo. Exemplo, FPM:
    até 2017  1.7.2.1.01.02.00  Cota-Parte do Fundo de Participação dos Municípios
    2018-2021 1.7.1.8.01.2.0    (cota mensal) + 1.7.1.8.01.3.0 / .4.0 (parcelas de 1%), sem linha-mãe própria
    2022+     1.7.1.1.51.0.0    Cota-Parte do FPM (linha-mãe) + filhas .1/.2/.3
Solução: cada conceito é identificado por (prefixo do código, expressão no rótulo). Entre as contas
que casam, ficam só as de nível mais alto (descarta-se a conta cuja "mãe" também casou) e somam-se
essas. Assim nunca se somam linhas hierárquicas em duplicidade (lição 2 do projeto).

Testado contra a base do trabalho anterior (2014-2023, Barreiras e Camaçari): ver
codigo/importa_base_original.py e data/original/validacao_extrator.csv.
"""
import re
import numpy as np
import pandas as pd

# conceito: (prefixo do código, regex no rótulo, regex de exclusão ou None)
CONCEITOS = {
    "iptu":            ("1.1", r"Predial e Territorial Urbana", None),
    "itbi":            ("1.1", r"Inter\W*Vivos", None),
    "iss":             ("1.1", r"Servi[çc]os de Qualquer Natureza|ISSQN", None),
    "fpm":             ("1.7", r"Fundo de Participa[çc][ãa]o d[oe]s? Munic|\bFPM\b", None),
    "itr":             ("1.7", r"Territorial Rural", None),
    "icms_cota":       ("1.7", r"Cota-Parte do ICMS|Cota-Parte do Imposto sobre Opera[çc]", r"Desonera"),
    "ipva_cota":       ("1.7", r"Cota-Parte do IPVA|Propriedade de Ve[íi]culos", None),
    "ipi_exp_cota":    ("1.7", r"Cota-Parte do IPI", None),
    "iof_ouro":        ("1.7", r"IOF.{0,5}Ouro|Ouro.{0,40}Ativo Financeiro", None),
    "fundeb":          ("1.7", r"FUNDEB|Manuten[çc][ãa]o e Desenvolvimento da Educa[çc][ãa]o B[áa]sica", None),
    "outras_multigov": ("1.7", r"Outras Transfer[êe]ncias Multigovernamentais", None),
    "sus":             ("1.7", r"Sistema [ÚU]nico de Sa[úu]de", None),
}

# Transferências não condicionadas (incondicionais) usadas como T no modelo flypaper.
# FUNDEB, SUS, FNDE e convênios ficam de fora por serem vinculados.
INCONDICIONAIS = ["fpm", "itr", "icms_cota", "ipva_cota", "ipi_exp_cota", "iof_ouro"]
ARRECADACAO_PROPRIA_IMPOSTOS = ["iptu", "itbi", "iss"]

_COD = re.compile(r"^\s*(?:RO|DO)?\s*(\d+(?:\.\d+)+)")


def separa_codigo(texto):
    """'1.7.1.8.01.2.0 Cota-Parte...' -> ('1.7.1.8.01.2.0', 'Cota-Parte...')."""
    if not isinstance(texto, str):
        return None, ""
    m = _COD.match(texto)
    if not m:
        return None, texto.strip()
    return m.group(1), texto[m.end():].lstrip(" -–").strip()


def significativo(codigo):
    """Segmentos sem os zeros finais: '1.7.2.1.01.02.00' -> (1,7,2,1,1,2)."""
    seg = [int(x) for x in codigo.split(".")]
    while seg and seg[-1] == 0:
        seg.pop()
    return tuple(seg)


def _topo(cods):
    """Mantém só os códigos que não descendem de outro código da lista."""
    sig = {c: significativo(c) for c in cods}
    out = []
    for c in cods:
        s = sig[c]
        if not any(o != c and len(sig[o]) < len(s) and s[:len(sig[o])] == sig[o] for o in cods):
            out.append(c)
    return out


def extrai(contas):
    """
    contas: DataFrame com colunas 'codigo' (str '1.7.2.1.01.02.00'), 'rotulo' (str) e 'valor' (float),
            de UM município, UM ano e UMA coluna da DCA (ex.: Receitas Brutas Realizadas).
    Retorna dict conceito -> valor (NaN se nenhuma conta casar: ausente ≠ zero) e
            dict conceito -> lista dos códigos usados (para auditoria).
    """
    c = contas.dropna(subset=["codigo"]).drop_duplicates(subset=["codigo"])
    valores, usados = {}, {}
    for nome, (prefixo, rx, rx_exc) in CONCEITOS.items():
        alvo = c[c.codigo.str.startswith(prefixo + ".") | (c.codigo == prefixo)]
        alvo = alvo[alvo.rotulo.str.contains(rx, case=False, regex=True, na=False)]
        if rx_exc:
            alvo = alvo[~alvo.rotulo.str.contains(rx_exc, case=False, regex=True, na=False)]
        if alvo.empty:
            valores[nome], usados[nome] = np.nan, []
            continue
        topo = _topo(list(alvo.codigo))
        sub = alvo[alvo.codigo.isin(topo)]
        valores[nome] = float(pd.to_numeric(sub.valor, errors="coerce").sum(min_count=1))
        usados[nome] = topo
    inc = [valores[k] for k in INCONDICIONAIS]
    # Soma das incondicionais: NaN só se TODAS faltarem (componentes pequenos como IOF-ouro podem não existir)
    valores["transf_incondicionais"] = float(np.nansum(inc)) if not all(np.isnan(inc)) else np.nan
    return valores, usados


def de_dca_api(df_dca):
    """Adapta o formato da API do Siconfi (cod_conta 'RO1.7...', conta '...') para extrai()."""
    out = df_dca.copy()
    cod_rot = out["conta"].apply(separa_codigo)
    out["codigo"] = out["cod_conta"].astype(str).str.replace(r"^(RO|DO)", "", regex=True)
    out.loc[~out["codigo"].str.match(r"^\d+(\.\d+)+$"), "codigo"] = None
    out["rotulo"] = [r for _, r in cod_rot]
    return out[["codigo", "rotulo", "valor"]]


In [ ]:
%%writefile codigo/coleta_municipios.py
"""
P2 - Finanças do Setor Público (UnB) - Coleta de dados brutos dos municípios
Municípios: definidos em config.py (Barreiras/BA e Camaçari/BA)

Uso:
    pip install requests pandas
    python coleta_municipios.py

Saídas (pasta data/raw/):
    siconfi_dca_bruto.csv    -> DCA completa (todos os anexos) por município e ano
    sidra_pib_municipal.csv  -> Tabela 5938 (PIB dos Municípios)
    sidra_populacao.csv      -> Tabela 6579 (estimativas de população)
    sidra_ipca_indice.csv    -> Tabela 1737 (IPCA, número-índice)
    log_coleta.csv           -> o que foi coletado ou falhou (documenta dados ausentes)

IMPORTANTE: não alterar os valores brutos. Tratamentos (deflação, per capita,
agregação por rubrica) devem ser feitos em outro script, em data/tratados/.
Nenhum valor ausente é substituído por zero.
"""
import os
import time
from datetime import datetime

import pandas as pd
import requests

from config import MUNICIPIOS, UF_ESPERADA, ANO_INI, ANO_FIM
ANOS = list(range(ANO_INI, ANO_FIM + 1))  # confira até qual ano a DCA está completa
SICONFI = "https://apidatalake.tesouro.gov.br/ords/siconfi/tt/dca"
SIDRA = "https://apisidra.ibge.gov.br/values"
OUT = os.path.join("data", "raw")
os.makedirs(OUT, exist_ok=True)

log = []


def registrar(fonte, detalhe, status, n=None):
    log.append({"data_hora": datetime.now().isoformat(timespec="seconds"),
                "fonte": fonte, "detalhe": detalhe, "status": status, "linhas": n})
    print(f"[{status}] {fonte} | {detalhe} | {n if n is not None else ''}")


def get_json(url, params=None, tentativas=4):
    for i in range(tentativas):
        try:
            r = requests.get(url, params=params, timeout=90)
            if r.status_code == 200:
                return r.json()
            erro = f"HTTP {r.status_code}"
        except Exception as e:  # rede, timeout, JSON inválido
            erro = repr(e)
        time.sleep(2 * (i + 1))
    raise RuntimeError(erro)


# ---------------------------------------------------------------- validação IBGE
def valida_municipios():
    """Aborta se o código IBGE não corresponder ao nome/UF esperados (evita coletar o município errado)."""
    for cod, nome in MUNICIPIOS.items():
        d = get_json(f"https://servicodados.ibge.gov.br/api/v1/localidades/municipios/{cod}")
        d = d[0] if isinstance(d, list) else d
        nome_ibge = d["nome"]
        try:
            uf = d["microrregiao"]["mesorregiao"]["UF"]["sigla"]
        except (KeyError, TypeError):
            uf = d["regiao-imediata"]["regiao-intermediaria"]["UF"]["sigla"]
        ok = nome_ibge == nome and uf == UF_ESPERADA[cod]
        registrar("IBGE localidades", f"{cod} -> {nome_ibge}/{uf}", "OK" if ok else "DIVERGENTE")
        if not ok:
            raise SystemExit(f"Código {cod} é {nome_ibge}/{uf}, esperado {nome}/{UF_ESPERADA[cod]}. Corrija config.py.")


# ---------------------------------------------------------------- Siconfi DCA
def coleta_dca():
    frames = []
    for ente, nome in MUNICIPIOS.items():
        for ano in ANOS:
            itens, offset = [], 0
            try:
                while True:
                    d = get_json(SICONFI, {"an_exercicio": ano, "id_ente": ente,
                                           "offset": offset})
                    itens += d.get("items", [])
                    if not d.get("hasMore"):
                        break
                    offset += d.get("limit", len(d.get("items", []))) or 0
                    time.sleep(1)  # a API do Siconfi limita a frequência de requisições
                if itens:
                    df = pd.DataFrame(itens)
                    df["municipio"] = nome
                    frames.append(df)
                    registrar("Siconfi DCA", f"{nome} {ano}", "OK", len(df))
                else:
                    registrar("Siconfi DCA", f"{nome} {ano}", "SEM DADOS (não entregue ou não publicado)", 0)
            except Exception as e:
                registrar("Siconfi DCA", f"{nome} {ano}", f"ERRO {e}")
            time.sleep(1)  # pausa entre anos (limite de requisições do Siconfi)
    if frames:
        pd.concat(frames, ignore_index=True).to_csv(
            os.path.join(OUT, "siconfi_dca_bruto.csv"), index=False, encoding="utf-8-sig")


# ---------------------------------------------------------------- IBGE SIDRA
def sidra(tabela, caminho, arquivo, descricao):
    url = f"{SIDRA}{caminho}"
    try:
        d = get_json(url)
        df = pd.DataFrame(d[1:])           # a linha 0 é o cabeçalho descritivo
        df.columns = [d[0].get(c, c) for c in df.columns]
        # Mantém o valor original ("V") e cria coluna numérica; '...', '-', 'X' viram NaN, nunca 0
        if "Valor" in df.columns:
            df["valor_num"] = pd.to_numeric(df["Valor"], errors="coerce")
        df.to_csv(os.path.join(OUT, arquivo), index=False, encoding="utf-8-sig")
        registrar("IBGE SIDRA", f"{descricao} (tabela {tabela})", "OK", len(df))
    except Exception as e:
        registrar("IBGE SIDRA", f"{descricao} (tabela {tabela})", f"ERRO {e}")


def coleta_ibge():
    cods = ",".join(str(c) for c in MUNICIPIOS)
    # PIB municipal: todas as variáveis, todos os períodos disponíveis
    sidra(5938, f"/t/5938/n6/{cods}/v/all/p/all", "sidra_pib_municipal.csv", "PIB dos Municípios")
    # População estimada (série 2001-). Para 2022 existe também o Censo (tabela 4709)
    sidra(6579, f"/t/6579/n6/{cods}/v/all/p/all", "sidra_populacao.csv", "Estimativas de população")
    # Censo 2022 (população residente) - usado onde a tab. 6579 não traz 2022
    sidra(4714, f"/t/4714/n6/{cods}/v/93/p/2022", "sidra_censo2022.csv", "Censo 2022 população")
    # IPCA número-índice (dez/1993=100), Brasil, mensal - base para o deflator
    sidra(1737, "/t/1737/n1/all/v/2266/p/all", "sidra_ipca_indice.csv", "IPCA número-índice")


if __name__ == "__main__":
    valida_municipios()
    coleta_dca()
    coleta_ibge()
    pd.DataFrame(log).to_csv(os.path.join(OUT, "log_coleta.csv"), index=False, encoding="utf-8-sig")
    print("\nConcluído. Veja data/raw/log_coleta.csv")


In [ ]:
%%writefile codigo/coleta_hungria.py
"""
P2 - Finanças do Setor Público (UnB) - Dados nacionais da Hungria (e Brasil, como referência)
Fonte: World Bank - World Development Indicators (API pública, metadados verificáveis)

Uso (Colab ou local):
    pip install requests pandas
    python coleta_hungria.py

Saídas (data/raw/):
    wdi_hungria_brasil_bruto.csv   -> série longa (país, ano, indicador, valor)
    wdi_metadados_indicadores.csv  -> nome, definição, fonte original e data de atualização
    eurostat_<dataset>_HU.csv      -> Eurostat em formato longo (códigos, rótulos, valor, flag)
    log_coleta_hungria.csv         -> cobertura (anos com dado) por indicador

ATENÇÃO: o WDI traz o governo CENTRAL. Os governos locais húngaros (setor S1313: receita, despesa,
transferências recebidas, impostos, despesa por função) vêm do Eurostat (arquivos eurostat_*_HU.csv). Indicadores sem dados
permanecem vazios (NaN); nenhum valor ausente é trocado por zero.
"""
import os
import time
from datetime import date

import pandas as pd
import requests

OUT = os.path.join("data", "raw")
os.makedirs(OUT, exist_ok=True)

from config import PAIS_ISO3, PAIS_NOME, REFERENCIA_ISO3, EUROSTAT_GEO, USAR_EUROSTAT, ANO_INI, ANO_FIM
PAISES = {PAIS_ISO3: PAIS_NOME, REFERENCIA_ISO3: "Brasil"}
ANOS = f"{ANO_INI - 2}:{ANO_FIM}"

INDICADORES = {
    # Escala e preços
    "NY.GDP.MKTP.CN": "pib_nominal_moeda_local",
    "NY.GDP.MKTP.KN": "pib_real_moeda_local_constante",
    "NY.GDP.PCAP.KN": "pib_per_capita_real_moeda_local",
    "SP.POP.TOTL": "populacao_total",
    "FP.CPI.TOTL": "ipc_indice_2010_100",
    "FP.CPI.TOTL.ZG": "inflacao_ipc_pct_anual",
    "PA.NUS.FCRF": "taxa_cambio_oficial_lcu_por_usd",
    # Governo central (NÃO é nível subnacional)
    "GC.REV.XGRT.GD.ZS": "receita_gov_central_exc_doacoes_pct_pib",
    "GC.TAX.TOTL.GD.ZS": "receita_tributaria_gov_central_pct_pib",
    "GC.XPN.TOTL.GD.ZS": "despesa_gov_central_pct_pib",
    "GC.XPN.TRFT.ZS": "subsidios_e_outras_transf_pct_despesa",
    "GC.XPN.COMP.ZS": "remuneracao_servidores_pct_despesa",
    "NE.CON.GOVT.ZS": "consumo_governo_geral_pct_pib",
    "SE.XPD.TOTL.GD.ZS": "gasto_educacao_pct_pib",
}

log = []


def get_json(url, params=None, tentativas=4):
    ultimo = None
    for i in range(tentativas):
        try:
            r = requests.get(url, params=params, timeout=60)
            if r.status_code == 200:
                return r.json()
            ultimo = f"HTTP {r.status_code}"
        except Exception as e:
            ultimo = repr(e)
        time.sleep(2 * (i + 1))
    raise RuntimeError(ultimo)


def coleta_series():
    linhas = []
    for iso, nome_pais in PAISES.items():
        for cod, nome_var in INDICADORES.items():
            url = f"https://api.worldbank.org/v2/country/{iso}/indicator/{cod}"
            try:
                d = get_json(url, {"format": "json", "date": ANOS, "per_page": 200})
                itens = d[1] if isinstance(d, list) and len(d) > 1 and d[1] else []
                n_ok = 0
                for it in itens:
                    v = it.get("value")
                    linhas.append({"pais_iso3": iso, "pais": nome_pais, "ano": int(it["date"]),
                                   "indicador_codigo": cod, "variavel": nome_var,
                                   "valor": float(v) if v is not None else None})
                    n_ok += v is not None
                log.append({"pais": iso, "indicador": cod, "status": "OK",
                            "anos_com_dado": n_ok, "anos_consultados": len(itens)})
                print(f"[OK] {iso} {cod}: {n_ok}/{len(itens)} anos com dado")
            except Exception as e:
                log.append({"pais": iso, "indicador": cod, "status": f"ERRO {e}",
                            "anos_com_dado": None, "anos_consultados": None})
                print(f"[ERRO] {iso} {cod}: {e}")
    pd.DataFrame(linhas).to_csv(os.path.join(OUT, "wdi_hungria_brasil_bruto.csv"),
                                index=False, encoding="utf-8-sig")


def coleta_metadados():
    meta = []
    for cod in INDICADORES:
        try:
            d = get_json(f"https://api.worldbank.org/v2/indicator/{cod}", {"format": "json"})
            info = d[1][0]
            meta.append({"indicador_codigo": cod, "nome": info.get("name"),
                         "definicao": info.get("sourceNote"),
                         "fonte_base": (info.get("source") or {}).get("value"),
                         "organizacao_fonte": info.get("sourceOrganization"),
                         "ultima_atualizacao_base": d[0].get("lastupdated"),
                         "data_acesso": date.today().isoformat()})
        except Exception as e:
            meta.append({"indicador_codigo": cod, "nome": f"ERRO {e}"})
    pd.DataFrame(meta).to_csv(os.path.join(OUT, "wdi_metadados_indicadores.csv"),
                              index=False, encoding="utf-8-sig")


EUROSTAT = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"
# Em vez de adivinhar códigos de item, baixa o recorte inteiro (todas as categorias das dimensões não
# filtradas) e salva em formato longo com códigos E rótulos. O tratamento escolhe as linhas depois.
#   gov_10a_main : receitas/despesas do governo por setor (S13 geral, S1311 central, S1313 local)
#   gov_10a_exp  : despesa por função (COFOG) - comparável à despesa municipal por função
#   gov_10a_taxag: impostos e contribuições por setor
#   prc_hicp_aind: inflação harmonizada (deflator da Hungria)
EUROSTAT_CONSULTAS = {
    "gov_10a_main":  {"sector": ["S13", "S1311", "S1313"], "unit": ["MIO_NAC", "PC_GDP"]},
    "gov_10a_exp":   {"sector": ["S13", "S1313"], "unit": ["MIO_NAC", "PC_GDP"], "na_item": ["TE"]},
    "gov_10a_taxag": {"sector": ["S13", "S1313"], "unit": ["MIO_NAC", "PC_GDP"]},
    "prc_hicp_aind": {"coicop": ["CP00"], "unit": ["INX_A_AVG", "RCH_A_AVG"]},
}


def jsonstat_longo(d):
    """Converte a resposta JSON-stat do Eurostat em linhas (uma por combinação de categorias)."""
    ids, tamanhos = d["id"], d["size"]
    cats = []
    for dim in ids:
        cat = d["dimension"][dim]["category"]
        pos = {v: k for k, v in cat["index"].items()}            # posição -> código
        cats.append((pos, cat.get("label", {})))
    linhas = []
    for chave, v in d.get("value", {}).items():
        resto, idx = int(chave), []
        for t in reversed(tamanhos):                             # índice plano -> posição em cada dimensão
            idx.append(resto % t); resto //= t
        idx.reverse()
        linha = {}
        for dim, (pos, rot), i in zip(ids, cats, idx):
            linha[dim] = pos[i]
            if dim not in ("time", "freq", "geo"):
                linha[f"{dim}_rotulo"] = rot.get(pos[i])
        linha["valor"] = v
        linha["flag"] = (d.get("status") or {}).get(chave)       # p = provisório, e = estimado, b = quebra
        linhas.append(linha)
    return linhas


def coleta_eurostat():
    for ds, filtros in EUROSTAT_CONSULTAS.items():
        params = [("format", "JSON"), ("lang", "EN"), ("geo", EUROSTAT_GEO), ("freq", "A"),
                  ("sinceTimePeriod", str(ANO_INI - 2))]
        for k, vals in filtros.items():
            params += [(k, v) for v in vals]
        try:
            d = get_json(f"{EUROSTAT}/{ds}", params)
            linhas = jsonstat_longo(d)
            df = pd.DataFrame(linhas)
            df.insert(0, "dataset", ds)
            df["atualizado_em"] = d.get("updated")
            df["data_acesso"] = date.today().isoformat()
            df.to_csv(os.path.join(OUT, f"eurostat_{ds}_HU.csv"), index=False, encoding="utf-8-sig")
            anos = sorted(df["time"].unique()) if len(df) else []
            log.append({"pais": EUROSTAT_GEO, "indicador": f"Eurostat {ds}", "status": "OK",
                        "anos_com_dado": f"{anos[0]}-{anos[-1]}" if anos else 0, "anos_consultados": len(df)})
            print(f"[OK] Eurostat {ds}: {len(df)} valores, anos {anos[:1]}...{anos[-1:]}")
        except Exception as e:
            log.append({"pais": EUROSTAT_GEO, "indicador": f"Eurostat {ds}", "status": f"ERRO {e}",
                        "anos_com_dado": None, "anos_consultados": None})
            print(f"[ERRO] Eurostat {ds}: {e}")


if __name__ == "__main__":
    coleta_series()
    if USAR_EUROSTAT:
        coleta_eurostat()
    coleta_metadados()
    pd.DataFrame(log).to_csv(os.path.join(OUT, "log_coleta_hungria.csv"),
                             index=False, encoding="utf-8-sig")
    print("\nConcluído. Veja data/raw/log_coleta_hungria.csv (cobertura por indicador).")


In [ ]:
%%writefile codigo/tratamento.py
"""
P2 - Finanças do Setor Público (UnB) - Tratamento dos dados dos municípios
Entrada : data/raw/   (gerado por coleta_municipios.py; NÃO é alterado)
Saída   : data/tratados/  (painel tratado + log de qualidade)

Decisões metodológicas (registrar no artigo e no dicionário de variáveis):
 - Receitas: DCA Anexo I-C, coluna "Receitas Brutas Realizadas" (sem somar linhas:
   cada variável é UMA conta exata da hierarquia). Deduções do FUNDEB em coluna própria.
 - Despesas: DCA Anexo I-D (natureza) e I-E (função), coluna "Despesas Empenhadas".
 - Deflator: média anual do IPCA (número-índice, SIDRA 1737), preços de ANO_BASE.
 - População: estimativa IBGE (SIDRA 6579). Anos ausentes (2022, 2023) são preenchidos de
   forma explícita e sinalizada (Censo 2022; interpolação linear em 2023), nunca com zero.
 - Dado ausente permanece NaN.
"""
import os
import numpy as np
import pandas as pd

from config import MUNICIPIOS as MUN, ANO_INI, ANO_FIM, ANO_BASE
from rubricas_receita import extrai, de_dca_api, CONCEITOS

RAW = os.path.join("data", "raw")
OUT = os.path.join("data", "tratados")
os.makedirs(OUT, exist_ok=True)

# Período, base de preços e municípios vêm de config.py (PIB municipal só existe até 2023).


def censo_2022():
    """Pop. do Censo 2022: baixada da SIDRA 4714 (raw) ou, se falhar, data/manual/censo2022.csv."""
    f1 = os.path.join(RAW, "sidra_censo2022.csv")
    f2 = os.path.join("data", "manual", "censo2022.csv")
    if os.path.exists(f1):
        c = pd.read_csv(f1)
        c = c.rename(columns={"Município (Código)": "cod_ibge"})
        c["populacao"] = pd.to_numeric(c["Valor"], errors="coerce")
        out = c.dropna(subset=["populacao"]).set_index("cod_ibge")["populacao"].to_dict()
    elif os.path.exists(f2):
        c = pd.read_csv(f2)
        out = c.set_index("cod_ibge")["populacao"].to_dict()
    else:
        raise FileNotFoundError("Sem Censo 2022: rode a coleta ou crie data/manual/censo2022.csv (cod_ibge,populacao)")
    faltam = [k for k in MUN if k not in out]
    if faltam:
        raise ValueError(f"Censo 2022 ausente para {faltam}")
    return {int(k): float(v) for k, v in out.items()}


CENSO_2022 = censo_2022()


# ----------------------------------------------------------------- helpers
def segs(cod, prefixo):
    """Segmentos numéricos de um cod_conta como 'RO1.7.0.0.00.0.0' ou 'DO3.0.00.00.00.00'.
    Funciona igual para o formato antigo (2015-2017: 'RO1.7.0.0.00.00.00') e o novo (2018+)."""
    if not isinstance(cod, str) or not cod.startswith(prefixo):
        return None
    try:
        return [int(x) for x in cod[len(prefixo):].split(".")]
    except ValueError:
        return None


def nivel(cod, prefixo, alvo):
    """True se o código é exatamente o nível `alvo` (ex.: [1,7]) com todo o resto zerado."""
    s = segs(cod, prefixo)
    return s is not None and s[:len(alvo)] == list(alvo) and all(v == 0 for v in s[len(alvo):])


def valor_unico(df, mascara, nome, ano, mun, log):
    """Retorna o valor de EXATAMENTE uma linha. Se houver 0 -> NaN; se >1 -> erro registrado."""
    sub = df[mascara]
    if len(sub) == 1:
        return float(sub["valor"].iloc[0])
    log.append({"municipio": mun, "ano": ano, "variavel": nome,
                "problema": f"{len(sub)} linhas encontradas (esperado 1)"})
    return np.nan


# ----------------------------------------------------------------- IPCA / deflator
def deflator():
    d = pd.read_csv(os.path.join(RAW, "sidra_ipca_indice.csv"))
    d["mes"] = d["Mês (Código)"].astype(int)
    d["ano"] = d["mes"] // 100
    d["idx"] = pd.to_numeric(d["Valor"], errors="coerce")
    d = d[d["idx"] > 0]
    g = d.groupby("ano")["idx"].agg(ipca_media_anual="mean", meses="count", ipca_dez="last")
    g = g[g["meses"] == 12].copy()                       # só anos completos
    g["fator_deflator"] = g.loc[ANO_BASE, "ipca_media_anual"] / g["ipca_media_anual"]
    return g.reset_index()


# ----------------------------------------------------------------- população
def populacao():
    p = pd.read_csv(os.path.join(RAW, "sidra_populacao.csv"))
    p = p.rename(columns={"Município (Código)": "cod_ibge", "Ano": "ano"})
    p["populacao"] = pd.to_numeric(p["Valor"], errors="coerce")
    p = p[["cod_ibge", "ano", "populacao"]].copy()
    p["pop_fonte"] = "IBGE estimativa (SIDRA 6579)"
    linhas = []
    for cod in MUN:
        s = p[p.cod_ibge == cod].set_index("ano")
        anos = range(ANO_INI, ANO_FIM + 1)
        for a in anos:
            if a in s.index and pd.notna(s.loc[a, "populacao"]):
                continue
            if a == 2022:
                linhas.append({"cod_ibge": cod, "ano": 2022, "populacao": CENSO_2022[cod],
                               "pop_fonte": "Censo 2022 (IBGE) - ausente na tab. 6579"})
            elif a == 2023 and 2024 in s.index:
                v = (CENSO_2022[cod] + s.loc[2024, "populacao"]) / 2
                linhas.append({"cod_ibge": cod, "ano": 2023, "populacao": v,
                               "pop_fonte": "INTERPOLADO linear (Censo 2022 e estimativa 2024)"})
    p = pd.concat([p, pd.DataFrame(linhas)], ignore_index=True)
    return p


# ----------------------------------------------------------------- PIB
def pib():
    d = pd.read_csv(os.path.join(RAW, "sidra_pib_municipal.csv"))
    d = d.rename(columns={"Município (Código)": "cod_ibge", "Ano": "ano", "Variável (Código)": "v"})
    d = d[d["Unidade de Medida"] == "Mil Reais"]
    d["valor_num"] = pd.to_numeric(d["Valor"], errors="coerce")
    nomes = {37: "pib_nominal", 543: "impostos_liq_subs", 498: "vab_total", 513: "vab_agro",
             517: "vab_industria", 6575: "vab_servicos", 525: "vab_adm_publica"}
    d = d[d["v"].isin(nomes)]
    w = d.pivot_table(index=["cod_ibge", "ano"], columns="v", values="valor_num", aggfunc="first")
    w = w.rename(columns=nomes) * 1000.0                # mil reais -> reais
    return w.reset_index()


# ----------------------------------------------------------------- DCA
def dca_rubricas(log):
    dca = pd.read_csv(os.path.join(RAW, "siconfi_dca_bruto.csv"))
    out = []
    for (cod, ano), g in dca.groupby(["cod_ibge", "exercicio"]):
        if not (ANO_INI <= ano <= ANO_FIM):
            continue
        mun = MUN[cod]
        r = g[g.anexo == "DCA-Anexo I-C"]
        rb = r[r.coluna == "Receitas Brutas Realizadas"]
        fu = r[r.coluna == "Deduções - FUNDEB"]
        od = r[r.coluna == "Outras Deduções da Receita"]
        d = g[g.anexo == "DCA-Anexo I-D"]
        db = d[d.coluna == "Despesas Empenhadas"]
        e = g[(g.anexo == "DCA-Anexo I-E") & (g.coluna == "Despesas Empenhadas")]

        linha = {"cod_ibge": cod, "municipio": mun, "ano": ano}
        # Receitas (níveis exatos da natureza de receita)
        # QUEBRA DE SÉRIE: até 2017 as transferências intergovernamentais ficam em 1.7.2
        # (1.7.2.1 = União; 1.7.2.2 = Estados). A partir de 2018: 1.7.1 = União; 1.7.2 = Estados.
        if ano <= 2017:
            uniao, estados = [1, 7, 2, 1], [1, 7, 2, 2]
        else:
            uniao, estados = [1, 7, 1], [1, 7, 2]
        niveis_rec = {"rec_corrente": [1, 0], "rec_tributaria": [1, 1], "rec_transf_correntes": [1, 7],
                      "rec_transf_uniao": uniao, "rec_transf_estados": estados}
        for nome, alvo in niveis_rec.items():
            m = rb.cod_conta.apply(lambda c: nivel(c, "RO", alvo))
            linha[nome] = valor_unico(rb, m, nome, ano, mun, log)
        # Rubricas detalhadas (FPM, cotas de ICMS/IPVA/IPI, ITR, IOF-ouro, IPTU, ISS, ITBI, FUNDEB, SUS...)
        # por código + rótulo, robusto às mudanças do ementário de 2018 e 2022 (ver rubricas_receita.py)
        det, _ = extrai(de_dca_api(rb))
        for k, x in det.items():
            linha[f"rec_{k}"] = x
        # Deduções (mesma conta, outras colunas)
        for nome, base in {"ded_fundeb_corrente": fu, "ded_outras_corrente": od}.items():
            m = base.cod_conta.apply(lambda c: nivel(c, "RO", [1, 0]))
            sub = base[m]
            linha[nome] = float(sub["valor"].iloc[0]) if len(sub) == 1 else np.nan
        # Receita total exceto intra (rótulo muda conforme o ano)
        # rec_total = linha 'TotalReceitas' (mesma definição em todos os anos; contém intra-orçamentárias,
        # assim como desp_total em I-D, o que mantém o resultado orçamentário consistente)
        linha["rec_total"] = valor_unico(rb, rb.cod_conta == "TotalReceitas", "rec_total", ano, mun, log)
        # Dedução do FUNDEB sobre a receita total (a receita bruta NÃO desconta o FUNDEB; a receita
        # líquida do balanço orçamentário desconta). Ausente => NaN (não é trocado por zero).
        linha["ded_fundeb_total"] = valor_unico(fu, fu.cod_conta == "TotalReceitas", "ded_fundeb_total", ano, mun, log)
        # Despesas por natureza (I-D)
        for nome, alvo in {"desp_corrente": [3, 0], "desp_pessoal": [3, 1], "desp_capital": [4, 0]}.items():
            m = db.cod_conta.apply(lambda c: nivel(c, "DO", alvo))
            linha[nome] = valor_unico(db, m, nome, ano, mun, log)
        m = (db.cod_conta == "TotalDespesas") & (db.conta == "Total Geral da Despesa")
        linha["desp_total"] = valor_unico(db, m, "desp_total", ano, mun, log)
        # Despesas por função (I-E): o código da função vem no texto da conta
        for nome, rot in {"desp_educacao": "12 - Educação", "desp_saude": "10 - Saúde"}.items():
            linha[nome] = valor_unico(e, e.conta == rot, nome, ano, mun, log)
        out.append(linha)
    return pd.DataFrame(out)


# ----------------------------------------------------------------- montagem
def main():
    log = []
    ipca = deflator()
    pop = populacao()
    pb = pib()
    rub = dca_rubricas(log)

    df = rub.merge(pop, on=["cod_ibge", "ano"], how="left") \
            .merge(pb, on=["cod_ibge", "ano"], how="left") \
            .merge(ipca[["ano", "ipca_media_anual", "fator_deflator"]], on="ano", how="left")

    # Sinal das deduções: a DCA registra as deduções como valores NEGATIVOS em alguns exercícios (formato antigo,
    # 2014-2016) e POSITIVOS nos demais. Padroniza como magnitude positiva antes de subtrair.
    for c in ["ded_fundeb_corrente", "ded_outras_corrente", "ded_fundeb_total"]:
        df[c] = df[c].abs()

    nominais = ["rec_corrente", "rec_tributaria", "rec_transf_correntes", "rec_transf_uniao",
                "rec_transf_estados", "ded_fundeb_corrente", "rec_total", "desp_corrente",
                "desp_pessoal", "desp_capital", "desp_total", "desp_educacao", "desp_saude", "pib_nominal"]
    nominais += [f"rec_{k}" for k in list(CONCEITOS) + ["transf_incondicionais"]]
    for c in nominais:
        df[f"{c}_real"] = df[c] * df["fator_deflator"]                    # reais de ANO_BASE
        df[f"{c}_real_pc"] = df[f"{c}_real"] / df["populacao"]            # per capita

    # Razões (sem unidade monetária): usam valores nominais do mesmo ano
    df["transf_sobre_rec_corrente"] = df["rec_transf_correntes"] / df["rec_corrente"]
    df["tributaria_sobre_rec_corrente"] = df["rec_tributaria"] / df["rec_corrente"]
    df["impostos_proprios"] = df[["rec_iptu", "rec_iss", "rec_itbi"]].sum(axis=1, min_count=3)  # NaN se faltar algum
    df["transf_incond_sobre_rec_corrente"] = df["rec_transf_incondicionais"] / df["rec_corrente"]
    # Receita corrente sem transferências extraordinárias (ex.: Barreiras 2017, R$ 178,6 mi em
    # "Outras Transferências Multigovernamentais"); usada em testes de sensibilidade
    df["rec_corrente_sem_extraord"] = df["rec_corrente"] - df["rec_outras_multigov"].fillna(0)
    df["transf_liq_fundeb"] = df["rec_transf_correntes"] - df["ded_fundeb_corrente"]
    df["desp_corrente_sobre_pib"] = df["desp_corrente"] / df["pib_nominal"]
    df["rec_tributaria_sobre_pib"] = df["rec_tributaria"] / df["pib_nominal"]
    df["transf_sobre_pib"] = df["rec_transf_correntes"] / df["pib_nominal"]
    df["educacao_sobre_desp_total"] = df["desp_educacao"] / df["desp_total"]
    # Receita líquida do FUNDEB e resultado orçamentário (corrigido: bruta - dedução FUNDEB - despesa empenhada)
    df["rec_total_liq_fundeb"] = df["rec_total"] - df["ded_fundeb_total"]
    df["resultado_orcamentario"] = df["rec_total_liq_fundeb"] - df["desp_total"]
    df["resultado_bruto_sem_deducao"] = df["rec_total"] - df["desp_total"]   # definição anterior, mantida para comparação

    df["regime_classif_receita"] = np.where(df["ano"] <= 2017, "ate_2017", "2018_em_diante")
    df["dummy_pos2018"] = (df["ano"] >= 2018).astype(int)
    df["ano_base_preco"] = ANO_BASE
    df = df.sort_values(["cod_ibge", "ano"]).reset_index(drop=True)
    df.to_csv(os.path.join(OUT, "tratado_municipios.csv"), index=False, encoding="utf-8-sig")

    # Log de qualidade: células vazias por variável + problemas de seleção
    vazios = df.isna().groupby(df["municipio"]).sum().T
    vazios = vazios[(vazios > 0).any(axis=1)]
    vazios.to_csv(os.path.join(OUT, "qualidade_celulas_vazias.csv"), encoding="utf-8-sig")
    pd.DataFrame(log).to_csv(os.path.join(OUT, "qualidade_selecao_contas.csv"), index=False, encoding="utf-8-sig")
    ipca.to_csv(os.path.join(OUT, "deflator_ipca.csv"), index=False, encoding="utf-8-sig")
    print("Linhas:", len(df), "| problemas de seleção:", len(log))
    print("Células vazias por variável:\n", vazios)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile codigo/tratamento_hungria.py
"""
P2 - Tratamento dos dados da Hungria (governos locais e governo geral) e do Brasil (referência)
Entrada : data/raw/eurostat_*_HU.csv e data/raw/wdi_hungria_brasil_bruto.csv (coleta_hungria.py)
Saída   : data/tratados/tratado_hungria.csv          painel anual (uma linha por ano)
          data/tratados/hungria_cofog_local.csv      despesa local por função (COFOG), % do PIB
          data/registro_alteracoes/hungria_P1_P2.csv comparação com a planilha da P1 (governo geral)

Definições (Eurostat, SEC 2010; setores S13 = governo geral, S1311 = central, S1313 = local):
  receita/despesa total              TR / TE
  transferências correntes recebidas D73REC  (transferências correntes dentro do governo geral)
  transferências de capital          D9REC   (inclui fundos da UE repassados)
  impostos locais                    D2REC + D5REC + D91REC
  deflator                           HICP médio anual (prc_hicp_aind, 2015=100), rebaseado para ANO_BASE
  população                          Banco Mundial, SP.POP.TOTL
Ausente permanece NaN. Valores monetários em milhões de HUF (MIO_NAC).
"""
import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from config import ANO_BASE  # noqa: E402

RAW = os.path.join("data", "raw")
OUT = os.path.join("data", "tratados")
REG = os.path.join("data", "registro_alteracoes")
os.makedirs(OUT, exist_ok=True); os.makedirs(REG, exist_ok=True)


def serie(m, setor, item, unidade):
    s = m[(m.sector == setor) & (m.na_item == item) & (m.unit == unidade)].set_index("time")["valor"]
    return s.astype(float)


def main():
    m = pd.read_csv(os.path.join(RAW, "eurostat_gov_10a_main_HU.csv"))
    anos = sorted(m.time.unique())
    df = pd.DataFrame(index=pd.Index(anos, name="ano"))

    for setor, pre in [("S1313", "loc"), ("S13", "gg"), ("S1311", "cen")]:
        for item, nome in [("TR", "receita"), ("TE", "despesa"), ("B9", "saldo")]:
            df[f"{pre}_{nome}_mio_huf"] = serie(m, setor, item, "MIO_NAC")
            df[f"{pre}_{nome}_pct_pib"] = serie(m, setor, item, "PC_GDP")
    df["loc_transf_correntes_mio_huf"] = serie(m, "S1313", "D73REC", "MIO_NAC")
    df["loc_transf_capital_mio_huf"] = serie(m, "S1313", "D9REC", "MIO_NAC")
    df["loc_impostos_mio_huf"] = sum(serie(m, "S1313", i, "MIO_NAC") for i in ["D2REC", "D5REC", "D91REC"])
    df["loc_transf_correntes_pct_pib"] = serie(m, "S1313", "D73REC", "PC_GDP")
    df["loc_impostos_pct_pib"] = sum(serie(m, "S1313", i, "PC_GDP") for i in ["D2REC", "D5REC", "D91REC"])

    # Razões (sem unidade): comparáveis às dos municípios brasileiros
    df["loc_transf_correntes_sobre_receita"] = df.loc_transf_correntes_mio_huf / df.loc_receita_mio_huf
    df["loc_transf_total_sobre_receita"] = (df.loc_transf_correntes_mio_huf + df.loc_transf_capital_mio_huf) / df.loc_receita_mio_huf
    df["loc_impostos_sobre_receita"] = df.loc_impostos_mio_huf / df.loc_receita_mio_huf
    df["loc_despesa_sobre_despesa_gg"] = df.loc_despesa_mio_huf / df.gg_despesa_mio_huf   # grau de descentralização

    # Deflator (HICP) e per capita
    h = pd.read_csv(os.path.join(RAW, "eurostat_prc_hicp_aind_HU.csv"))
    hicp = h[h.unit == "INX_A_AVG"].set_index("time")["valor"].astype(float)
    df["hicp_2015_100"] = hicp
    df["fator_deflator"] = hicp.get(ANO_BASE, np.nan) / df["hicp_2015_100"]
    w = pd.read_csv(os.path.join(RAW, "wdi_hungria_brasil_bruto.csv"))
    hun = w[w.pais_iso3 == "HUN"].pivot_table(index="ano", columns="indicador_codigo", values="valor")
    df["populacao"] = hun.get("SP.POP.TOTL")
    df["pib_mio_huf_wdi"] = hun.get("NY.GDP.MKTP.CN") / 1e6
    for c in ["loc_receita", "loc_despesa", "loc_transf_correntes", "loc_impostos"]:
        df[f"{c}_huf_real_pc"] = df[f"{c}_mio_huf"] * 1e6 * df["fator_deflator"] / df["populacao"]
    df["ano_base_preco"] = ANO_BASE
    df.reset_index().to_csv(os.path.join(OUT, "tratado_hungria.csv"), index=False, encoding="utf-8-sig")

    # Despesa local por função (COFOG, 1º nível), % do PIB e % da despesa local
    e = pd.read_csv(os.path.join(RAW, "eurostat_gov_10a_exp_HU.csv"))
    e = e[(e.sector == "S1313") & (e.na_item == "TE") & e.cofog99.str.match(r"^GF\d\d$")]
    cof = e.pivot_table(index=["cofog99", "cofog99_rotulo"], columns=["unit", "time"], values="valor")
    cof.to_csv(os.path.join(OUT, "hungria_cofog_local.csv"), encoding="utf-8-sig")

    # Comparação com a P1 (planilha Hungria_2014_2023.xlsx: governo geral, HUF bilhões; 2014-2021 estimados)
    p1 = {"Receita Total (HUF bilhões)": [15581, 15735, 16155, 17060, 18861, 21054, 22310, 24608, 28145, 32111],
          "Despesa Total (HUF bilhões)": [16486, 16383, 16766, 17864, 19705, 21866, 25932, 28526, 32271, 37146]}
    comp = []
    for i, ano in enumerate(range(2014, 2024)):
        for var, col in [("Receita Total (HUF bilhões)", "gg_receita_mio_huf"), ("Despesa Total (HUF bilhões)", "gg_despesa_mio_huf")]:
            v1, v2 = p1[var][i], df.loc[ano, col] / 1000
            comp.append({"ano": ano, "variavel_P1": var, "valor_P1": v1, "valor_P2_eurostat": round(v2, 1),
                         "dif_rel": (v2 - v1) / v1,
                         "motivo": ("P1: estimativa (% do PIB × PIB estimado)" if ano <= 2021 else "P1: KSH/EDP")
                                   + "; P2: Eurostat gov_10a_main, valor oficial em MIO_NAC"})
    pd.DataFrame(comp).to_csv(os.path.join(REG, "hungria_P1_P2.csv"), index=False, encoding="utf-8-sig")

    cols = ["loc_receita_pct_pib", "loc_despesa_pct_pib", "loc_transf_correntes_sobre_receita",
            "loc_impostos_sobre_receita", "loc_despesa_sobre_despesa_gg", "loc_despesa_huf_real_pc"]
    print(df[cols].round(3).to_string())
    print(pd.DataFrame(comp).round(3).to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile codigo/importa_base_original.py
"""
Importa a base do TRABALHO ANTERIOR (P1, João Paulo Cortes da Cunha Dutra, 241021838) para formato
longo/tabular, sem alterar os arquivos originais, e valida o extrator de rubricas contra ela.

Fonte: https://github.com/Jhons777/P1-Finan-as-do-Setor-Publico---Jo-o-Paulo-Cortes-Da-Cunha-Dutra-241021838
Commit usado: 4ae3ff41d0895097b752359aad152a4cac551f33 (08/05/2026)

Uso:  python importa_base_original.py data/original/P1_anterior
Saída (em data/original/):
  contas_receita_original.csv   todas as contas de receita das abas brutas (mun, ano, código, rótulo, valor)
  painel_original.csv           painel nominal da P1 (receitas, despesas, funções, população, PIB)
  validacao_extrator.csv        extrator novo × colunas da P1 (diferença absoluta e relativa)
  achados_auditoria.csv         inconsistências detectadas automaticamente
"""
import os
import re
import sys
import numpy as np
import pandas as pd
import openpyxl

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from rubricas_receita import extrai, separa_codigo  # noqa: E402

ORIG = sys.argv[1] if len(sys.argv) > 1 else os.path.join("data", "original", "P1_anterior")
OUT = os.path.join("data", "original")
os.makedirs(OUT, exist_ok=True)
COD = {"Barreiras": 2903201, "Camaçari": 2905701}
ABA = re.compile(r"^\s*(Barreiras|Camaçari)\s*-\s*BA\s*\((\d{4})\)\s*$")


def mun_nome(txt):
    return "Barreiras" if str(txt).startswith("Barreiras") else "Camaçari"


def contas_brutas(arquivo):
    wb = openpyxl.load_workbook(arquivo, read_only=True, data_only=True)
    linhas = []
    for nome in wb.sheetnames:
        m = ABA.match(nome)
        if not m:
            continue
        mun, ano = m.group(1), int(m.group(2))
        for r in wb[nome].iter_rows(values_only=True):
            if len(r) < 3 or r[1] is None:
                continue
            codigo, rotulo = separa_codigo(str(r[1]))
            if codigo is None and not str(r[1]).upper().startswith(("TOTAL", "RECEITAS (EXCETO")):
                continue
            valor = pd.to_numeric(r[2], errors="coerce")
            linhas.append({"municipio": mun, "cod_ibge": COD[mun], "ano": ano,
                           "codigo": codigo, "rotulo": rotulo if codigo else str(r[1]), "valor": valor})
    return pd.DataFrame(linhas)


def main():
    rd = os.path.join(ORIG, "Dados Brutos")
    rec = contas_brutas(os.path.join(rd, "Receitas orçamentarias", "Base_receitas (Principal).xlsx"))
    rec.to_csv(os.path.join(OUT, "contas_receita_original.csv"), index=False, encoding="utf-8-sig")

    # ---------------- painel da P1 (valores nominais, como publicados)
    br = pd.read_excel(os.path.join(rd, "Receitas orçamentarias", "Base_receitas (Principal).xlsx"), "Base_Receitas")
    bd = pd.read_excel(os.path.join(rd, "Despesas orçamentarias", "base_despesas (principal).xlsx"), "Base_Despesas")
    bf = pd.read_excel(os.path.join(rd, "Despesas por função", "base_despesas_funcao_2014_2023.xlsx"), "Base_Despesas_Funcao")
    bp = pd.read_excel(os.path.join(rd, "Base_Populacao_Municipal.xlsx"), "Base_Populacao")
    bpib = pd.read_excel(os.path.join(rd, "Base_PIB_corrigida.xlsx"), "Base_PIB")
    for d in (br, bd, bf, bp, bpib):
        d["municipio"] = d["Município"].map(mun_nome)
    keep = lambda d, cols: d[["Ano", "municipio"] + cols]  # noqa: E731
    painel = keep(br, ["Receita_Total", "Receita_Tributaria", "IPTU", "ISS", "ITBI", "Transferencias", "FPM",
                       "ICMS", "FUNDEB", "Receita_Capital"])
    painel = painel.merge(keep(bd, [c for c in bd.columns if c.startswith(("Despesa", "Pessoal", "Juros", "Outras",
                                                                         "Investimentos", "Inversoes", "Amortizacao"))]),
                          on=["Ano", "municipio"], how="outer")
    painel = painel.merge(keep(bf, ["Educacao", "Saude", "Saneamento", "Urbanismo", "Assistencia_Social",
                                    "Administracao"]).add_prefix("Funcao_").rename(
        columns={"Funcao_Ano": "Ano", "Funcao_municipio": "municipio"}), on=["Ano", "municipio"], how="outer")
    painel = painel.merge(keep(bp, ["População"]), on=["Ano", "municipio"], how="outer")
    painel = painel.merge(keep(bpib, ["PIB", "PIB_per_capita"]), on=["Ano", "municipio"], how="outer")
    painel = painel.rename(columns={"Ano": "ano"}).sort_values(["municipio", "ano"])
    painel["cod_ibge"] = painel["municipio"].map(COD)
    painel.to_csv(os.path.join(OUT, "painel_original.csv"), index=False, encoding="utf-8-sig")

    # ---------------- validação do extrator
    comp = {"IPTU": "iptu", "ISS": "iss", "ITBI": "itbi", "FPM": "fpm", "ICMS": "icms_cota", "FUNDEB": "fundeb"}
    val, usados_log = [], []
    for (mun, ano), g in rec[rec.codigo.notna()].groupby(["municipio", "ano"]):
        v, usados = extrai(g)
        p1 = br[(br.municipio == mun) & (br.Ano == ano)]
        for col, conc in comp.items():
            a = float(p1[col].iloc[0]) if len(p1) else np.nan
            val.append({"municipio": mun, "ano": ano, "variavel": conc, "valor_P1": a, "valor_extrator": v[conc],
                        "dif_abs": v[conc] - a, "dif_rel": (v[conc] - a) / a if a else np.nan,
                        "contas_usadas": ";".join(usados[conc])})
        for k in ["itr", "ipva_cota", "ipi_exp_cota", "iof_ouro", "outras_multigov", "sus", "transf_incondicionais"]:
            val.append({"municipio": mun, "ano": ano, "variavel": k, "valor_P1": np.nan, "valor_extrator": v[k],
                        "dif_abs": np.nan, "dif_rel": np.nan, "contas_usadas": ";".join(usados.get(k, []))})
    val = pd.DataFrame(val)
    val.to_csv(os.path.join(OUT, "validacao_extrator.csv"), index=False, encoding="utf-8-sig")

    # ---------------- achados automáticos
    ach = []
    for c in [c for c in painel.columns if c.startswith("Funcao_")]:
        for _, r in painel[painel[c] == 0].iterrows():
            ach.append({"tipo": "zero_no_lugar_de_ausente", "municipio": r.municipio, "ano": r.ano, "variavel": c,
                        "detalhe": ("P1 gravou 0, mas a função não consta na aba bruta do ano. Tratado como ausente (NaN). "
                                   "Saúde/Administração: certamente lacuna de coleta; Saneamento: pode ser gasto nulo "
                                   "(a DCA omite funções sem empenho) - conferir na nova coleta")})
    for _, r in painel[painel["População"].isna()].iterrows():
        ach.append({"tipo": "populacao_ausente", "municipio": r.municipio, "ano": r.ano, "variavel": "População",
                    "detalhe": "P1 não preencheu; per capita do ano ficou vazio"})
    fp = val[(val.variavel == "fpm") & (val.dif_rel.abs() >= 1e-3)]
    for _, r in fp.iterrows():
        ach.append({"tipo": "definicao_inconsistente", "municipio": r.municipio, "ano": r.ano, "variavel": "FPM",
                    "detalhe": (f"P1 = R$ {r.valor_P1:,.2f} (só cota mensal); com as parcelas de 1% = "
                                f"R$ {r.valor_extrator:,.2f}. A partir de 2018 a P1 soma as parcelas: quebra de definição")})
    om = val[(val.variavel == "outras_multigov") & (val.valor_extrator > 0)]
    for _, r in om.iterrows():
        ach.append({"tipo": "transferencia_extraordinaria", "municipio": r.municipio, "ano": r.ano,
                    "variavel": "Outras Transferências Multigovernamentais",
                    "detalhe": f"R$ {r.valor_extrator:,.2f} incluídos em Receita_Total/Transferencias da P1"})
    for mun, g in painel.groupby("municipio"):
        g = g.sort_values("ano")
        var = g["Receita_Total"].pct_change()
        for ano, x in zip(g.ano, var):
            if pd.notna(x) and abs(x) > 0.25:
                ach.append({"tipo": "salto_receita_total", "municipio": mun, "ano": ano, "variavel": "Receita_Total",
                            "detalhe": f"variação nominal de {x:+.1%} em relação ao ano anterior"})
    pd.DataFrame(ach).to_csv(os.path.join(OUT, "achados_auditoria.csv"), index=False, encoding="utf-8-sig")

    v6 = val[val.variavel.isin(comp.values())]
    print("Contas de receita lidas:", len(rec))
    print("Validação (|dif_rel| < 0,1%):", (v6.dif_rel.abs() < 1e-3).sum(), "de", v6.dif_rel.notna().sum())
    print(v6[v6.dif_rel.abs() >= 1e-3][["municipio", "ano", "variavel", "valor_P1", "valor_extrator", "dif_rel"]]
          .to_string(index=False))
    print("Achados de auditoria:", len(ach))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile codigo/compara_original_atualizado.py
"""
Registro de alterações: compara a base da P1 (data/original/painel_original.csv, gerada por
importa_base_original.py) com o painel tratado da P2 (data/tratados/tratado_municipios.csv),
ano a ano, nos anos em comum (2014-2023), em VALORES NOMINAIS (mesma unidade da P1).

Saídas (data/registro_alteracoes/):
  comparacao_P1_P2.csv   uma linha por município × ano × variável: valor P1, valor P2, diferença e motivo
  resumo_alteracoes.csv  contagem por variável e tipo de alteração
Uso: python codigo/compara_original_atualizado.py
"""
import os
import numpy as np
import pandas as pd

OUT = os.path.join("data", "registro_alteracoes")
os.makedirs(OUT, exist_ok=True)

# variável da P1 -> (variável da P2, fator para levar a P1 à unidade da P2)
MAPA = {
    "Receita_Total": ("rec_total", 1), "Receita_Tributaria": ("rec_tributaria", 1),
    "IPTU": ("rec_iptu", 1), "ISS": ("rec_iss", 1), "ITBI": ("rec_itbi", 1),
    "Transferencias": ("rec_transf_correntes", 1), "FPM": ("rec_fpm", 1), "ICMS": ("rec_icms_cota", 1),
    "FUNDEB": ("rec_fundeb", 1), "Despesa_Total": ("desp_total", 1), "Despesas_Correntes": ("desp_corrente", 1),
    "Pessoal_Encargos": ("desp_pessoal", 1), "Despesas_Capital": ("desp_capital", 1),
    "Funcao_Educacao": ("desp_educacao", 1), "Funcao_Saude": ("desp_saude", 1),
    "População": ("populacao", 1), "PIB": ("pib_nominal", 1000),   # P1: PIB em R$ mil
}
TOL = 1e-3   # diferença relativa abaixo de 0,1% = igual (arredondamento)


def motivo(var_p1, mun, ano, v1, v2, dca):
    if pd.isna(v1) and pd.notna(v2):
        if var_p1 == "População":
            return "PREENCHIDO", "P1 sem valor; P2 usa Censo 2022 (2022) e interpolação linear Censo 2022-estimativa 2024 (2023)"
        return "PREENCHIDO", "P1 sem valor; P2 coletou na fonte"
    if pd.notna(v1) and v1 == 0 and pd.notna(v2) and v2 > 0:
        return "CORRIGIDO", "P1 gravou zero (função ausente na aba bruta); P2 traz o valor da DCA, Anexo I-E"
    if var_p1 == "FPM":
        return "CORRIGIDO", "P1 contou só a cota mensal; P2 soma também as parcelas de 1% (mesma definição em todos os anos)"
    if var_p1 == "Receita_Total" and dca is not None:
        intra = dca.get((mun, ano))
        if intra and abs((v2 - v1) - intra) / intra < TOL:
            return "CORRIGIDO", (f"P1 usou receita exceto intra-orçamentárias (R$ {intra:,.2f} a menos), mas despesa total "
                                 "com intra; P2 usa o total com intra nas duas, mantendo o resultado consistente")
    if var_p1 == "População":
        return "CORRIGIDO", "Valor da P1 diverge da estimativa oficial do IBGE (SIDRA 6579); provável erro de digitação na P1"
    return "A INVESTIGAR", ""


def intra_orcamentarias():
    """Receita intra-orçamentária por município e ano (para explicar diferenças da receita total)."""
    f = os.path.join("data", "raw", "siconfi_dca_bruto.csv")
    if not os.path.exists(f):
        return None
    d = pd.read_csv(f, low_memory=False)
    d = d[(d.anexo == "DCA-Anexo I-C") & (d.coluna == "Receitas Brutas Realizadas")
          & (d.cod_conta == "ReceitasIntraOrcamentarias")]
    return {(m, int(a)): float(v) for m, a, v in zip(d.municipio, d.exercicio, d.valor)}


def main():
    p1 = pd.read_csv(os.path.join("data", "original", "painel_original.csv"))
    p2 = pd.read_csv(os.path.join("data", "tratados", "tratado_municipios.csv"))
    dca = intra_orcamentarias()
    x = p1.merge(p2, on=["municipio", "ano"], how="inner")
    x = x[x.ano <= 2023]                     # anos cobertos pela P1 (2024 na P1 só tem população)
    linhas = []
    for var_p1, (var_p2, fator) in MAPA.items():
        for _, r in x.iterrows():
            v1 = r[var_p1] * fator if pd.notna(r[var_p1]) else np.nan
            v2 = r[var_p2]
            if pd.isna(v1) and pd.isna(v2):
                tipo, mot = "AUSENTE NAS DUAS", "sem dado na fonte"
            elif pd.notna(v1) and pd.notna(v2) and v1 != 0 and abs(v2 - v1) / abs(v1) < TOL:
                tipo, mot = "IGUAL", ""
            else:
                tipo, mot = motivo(var_p1, r.municipio, r.ano, v1, v2, dca)
            linhas.append({"municipio": r.municipio, "ano": int(r.ano), "variavel_P1": var_p1, "variavel_P2": var_p2,
                           "valor_P1": v1, "valor_P2": v2,
                           "dif_abs": v2 - v1 if pd.notna(v1) and pd.notna(v2) else np.nan,
                           "dif_rel": (v2 - v1) / v1 if pd.notna(v1) and pd.notna(v2) and v1 else np.nan,
                           "tipo": tipo, "motivo": mot})
    c = pd.DataFrame(linhas)
    c.to_csv(os.path.join(OUT, "comparacao_P1_P2.csv"), index=False, encoding="utf-8-sig")
    resumo = c.pivot_table(index="variavel_P1", columns="tipo", values="ano", aggfunc="count", fill_value=0)
    resumo.to_csv(os.path.join(OUT, "resumo_alteracoes.csv"), encoding="utf-8-sig")
    print(resumo)
    print("\nAlterações (exceto IGUAL):")
    print(c[c.tipo != "IGUAL"][["municipio", "ano", "variavel_P1", "valor_P1", "valor_P2", "dif_rel", "tipo"]]
          .to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile codigo/analise.py
"""
P2 - Análise: estatísticas descritivas, composição da receita, regressões do efeito flypaper,
indicadores da Hungria, gráficos e tabelas em LaTeX (prontas para o Overleaf).

Entradas: data/tratados/tratado_municipios.csv, data/tratados/tratado_hungria.csv,
          data/registro_alteracoes/comparacao_P1_P2.csv, data/registro_alteracoes/hungria_P1_P2.csv
Saídas:   artigo/tabelas/*.tex, artigo/figuras/*.pdf, resultados/*.csv
Uso:      python codigo/analise.py      (só numpy, pandas, scipy e matplotlib)

Regressão (MQO com efeito fixo de município, erros-padrão robustos HC1):
    ln G_it = a_i + bT ln T_it + bY ln Y_it + d D2018_t + e_it
G = despesa real per capita; T = transferências incondicionais reais per capita; Y = PIB real per capita.
O PIB municipal só existe até 2023, por isso a amostra da regressão termina em 2023.
Com N pequeno, os resultados são exploratórios (associação, não causalidade).
"""
import os
import sys
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from config import JANELA_ANALISE, ANO_BASE  # noqa: E402

TAB = os.path.join("artigo", "tabelas"); FIG = os.path.join("artigo", "figuras"); RES = "resultados"
for d in (TAB, FIG, RES):
    os.makedirs(d, exist_ok=True)
A0, A1 = JANELA_ANALISE
COR = {"Barreiras": "#2a78d6", "Camaçari": "#eb6834", "Hungria": "#1baf7a"}   # validado (dataviz)
TINTA, TINTA2, GRADE = "#1f1f1e", "#5f5e58", "#e4e3de"
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.edgecolor": TINTA2, "axes.labelcolor": TINTA,
                     "xtick.color": TINTA2, "ytick.color": TINTA2, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.color": GRADE, "grid.linewidth": 0.6,
                     "axes.axisbelow": True, "legend.frameon": False})


# ------------------------------------------------------------------ utilidades
def br(x, d=0):
    """Número no padrão brasileiro (1.234,5). NaN vira '--' (ausente, nunca zero)."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "--"
    s = f"{x:,.{d}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")


def pct(x, d=1):
    return "--" if pd.isna(x) else br(100 * x, d) + "\\%"


def tabela_tex(nome, legenda, rotulo, cabecalho, linhas, nota, alinhamento=None, fonte_peq=True):
    """Tabela booktabs; reduz automaticamente para caber na largura do texto (adjustbox) e põe a nota embaixo."""
    al = alinhamento or ("l" + "r" * (len(cabecalho) - 1))
    tex = ["\\begin{table}[H]", "\\centering" + ("\\footnotesize" if fonte_peq else ""),
           "\\setlength{\\tabcolsep}{4pt}", f"\\caption{{{legenda}}}", f"\\label{{{rotulo}}}",
           "\\begin{adjustbox}{max width=\\textwidth}", f"\\begin{{tabular}}{{{al}}}", "\\toprule",
           " & ".join(cabecalho) + " \\\\", "\\midrule"]
    for ln in linhas:
        tex.append(ln if isinstance(ln, str) else " & ".join(ln) + " \\\\")
    tex += ["\\bottomrule", "\\end{tabular}", "\\end{adjustbox}",
            "\\par\\vspace{3pt}\\parbox{\\textwidth}{\\scriptsize " + nota + "}", "\\end{table}"]
    with open(os.path.join(TAB, nome), "w", encoding="utf-8") as f:
        f.write("\n".join(tex) + "\n")


def mqo_hc1(y, X, nomes):
    """MQO com erros robustos HC1. Retorna dict com coeficientes, ep, p, N, R2, DW e matriz de covariância."""
    y, X = np.asarray(y, float), np.asarray(X, float)
    n, k = X.shape
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    V = n / (n - k) * XtX_inv @ (X.T * e ** 2) @ X @ XtX_inv
    ep = np.sqrt(np.diag(V))
    p = 2 * stats.t.sf(np.abs(b / ep), n - k)
    r2 = 1 - (e @ e) / np.sum((y - y.mean()) ** 2)
    dw = np.sum(np.diff(e) ** 2) / (e @ e)
    return {"b": dict(zip(nomes, b)), "ep": dict(zip(nomes, ep)), "p": dict(zip(nomes, p)), "n": n, "k": k,
            "r2": r2, "dw": dw, "V": pd.DataFrame(V, index=nomes, columns=nomes), "resid": e}


def wald_igualdade(r, a, b):
    """H0: coef[a] = coef[b] (teste t com covariância HC1)."""
    d = r["b"][a] - r["b"][b]
    var = r["V"].loc[a, a] + r["V"].loc[b, b] - 2 * r["V"].loc[a, b]
    t = d / np.sqrt(var)
    return d, 2 * stats.t.sf(abs(t), r["n"] - r["k"])


def estrelas(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""


# ------------------------------------------------------------------ dados
t = pd.read_csv(os.path.join("data", "tratados", "tratado_municipios.csv")).copy()
h = pd.read_csv(os.path.join("data", "tratados", "tratado_hungria.csv")).set_index("ano")
t["impostos_proprios_real_pc"] = t[["rec_iptu_real_pc", "rec_iss_real_pc", "rec_itbi_real_pc"]].sum(axis=1, min_count=3)
t["outras_incond_real_pc"] = t[["rec_itr_real_pc", "rec_ipva_cota_real_pc", "rec_ipi_exp_cota_real_pc"]].sum(axis=1, min_count=1)
t["impostos_proprios_sobre_rc"] = t["impostos_proprios"] / t["rec_corrente"]
t["fpm_sobre_rc"] = t["rec_fpm"] / t["rec_corrente"]
t["icms_sobre_rc"] = t["rec_icms_cota"] / t["rec_corrente"]
t["transf_corr_sobre_rc_sem_extra"] = (t["rec_transf_correntes"] - t["rec_outras_multigov"].fillna(0)) / t["rec_corrente_sem_extraord"]
j = t[(t.ano >= A0) & (t.ano <= A1)].copy()
MUN = ["Barreiras", "Camaçari"]

# ------------------------------------------------------------------ Tabela: estatísticas descritivas
VARS = [("rec_corrente_real_pc", "Receita corrente"), ("impostos_proprios_real_pc", "Impostos próprios$^{b}$"),
        ("rec_transf_incondicionais_real_pc", "Transf. incondicionais$^{c}$"), ("rec_fpm_real_pc", "\\quad FPM"),
        ("rec_icms_cota_real_pc", "\\quad Cota-parte do ICMS"), ("rec_fundeb_real_pc", "FUNDEB"),
        ("desp_total_real_pc", "Despesa total"), ("desp_educacao_real_pc", "\\quad Educação"),
        ("desp_saude_real_pc", "\\quad Saúde"), ("pib_nominal_real_pc", "PIB per capita$^{a}$")]
linhas, desc = [], []
for v, rot in VARS:
    cel = [rot]
    for m in MUN:
        s = j.loc[j.municipio == m, v].dropna()
        cel += [br(s.mean()), br(s.std()), br(s.min()), br(s.max())]
        desc.append({"variavel": v, "municipio": m, "media": s.mean(), "dp": s.std(), "min": s.min(), "max": s.max(), "n": len(s)})
    linhas.append(cel)
pd.DataFrame(desc).to_csv(os.path.join(RES, "descritivas.csv"), index=False, encoding="utf-8-sig")
tabela_tex("tab_descritivas.tex",
           f"Estatísticas descritivas: valores reais per capita, Barreiras e Camaçari ({A0}--{A1})", "tab:descritivas",
           ["R\\$ de %d per capita" % ANO_BASE, "Média", "DP", "Mín.", "Máx.", "Média", "DP", "Mín.", "Máx."],
           linhas,
           (f"Nota: {A1 - A0 + 1} observações anuais por município, exceto (a) PIB, disponível até 2023 ({2023 - A0 + 1} anos). "
            f"Valores deflacionados pela média anual do IPCA (preços de {ANO_BASE}) e divididos pela população (estimativas do IBGE; "
            "Censo 2022 em 2022 e interpolação linear em 2023). (b) IPTU + ISS + ITBI. (c) FPM + ITR + cotas-partes de ICMS, "
            "IPVA e IPI-exportação + IOF-ouro. Despesa: empenhada. DP = desvio-padrão. Fonte: elaboração própria com dados do Siconfi (DCA) e do IBGE."),
           alinhamento="lrrrrrrrr")
# insere o cabeçalho de dois níveis (Barreiras / Camaçari)
f = os.path.join(TAB, "tab_descritivas.tex"); s = open(f, encoding="utf-8").read()
s = s.replace("\\toprule\n", "\\toprule\n & \\multicolumn{4}{c}{Barreiras} & \\multicolumn{4}{c}{Camaçari} \\\\\n"
              "\\cmidrule(lr){2-5}\\cmidrule(lr){6-9}\n", 1)
open(f, "w", encoding="utf-8").write(s)

# ------------------------------------------------------------------ Tabela: composição e dependência
ANOS_C = [a for a in [2016, 2019, 2022, 2025] if A0 <= a <= A1]
IND = [("fpm_sobre_rc", "FPM"), ("icms_sobre_rc", "Cota-parte do ICMS"),
       ("transf_incond_sobre_rec_corrente", "Transf. incondicionais"),
       ("transf_sobre_rec_corrente", "Transf. correntes (total)"),
       ("impostos_proprios_sobre_rc", "Impostos próprios"),
       ("tributaria_sobre_rec_corrente", "Receita tributária")]
linhas, comp = [], []
for v, rot in IND:
    cel = [rot]
    for m in MUN:
        for a in ANOS_C:
            x = t.loc[(t.municipio == m) & (t.ano == a), v]
            x = float(x.iloc[0]) if len(x) else np.nan
            cel.append(pct(x)); comp.append({"indicador": v, "municipio": m, "ano": a, "valor": x})
    linhas.append(cel)
pd.DataFrame(comp).to_csv(os.path.join(RES, "composicao_receita.csv"), index=False, encoding="utf-8-sig")
tabela_tex("tab_composicao.tex", "Composição da receita corrente: participação de cada fonte (\\% da receita corrente)",
           "tab:composicao", ["Fonte"] + [str(a) for a in ANOS_C] * 2, linhas,
           ("Nota: razões calculadas com valores nominais do mesmo ano. A receita corrente é bruta (inclui as transferências do "
            "FUNDEB antes da dedução). Em 2017, Barreiras registrou R\\$ 178,6 milhões em ``Outras Transferências "
            "Multigovernamentais'', que elevam a participação das transferências naquele ano (não exibido). Fonte: elaboração "
            "própria com dados do Siconfi (DCA, Anexo I-C)."), alinhamento="l" + "r" * (2 * len(ANOS_C)))
f = os.path.join(TAB, "tab_composicao.tex"); s = open(f, encoding="utf-8").read()
nA = len(ANOS_C)
s = s.replace("\\toprule\n", "\\toprule\n & \\multicolumn{%d}{c}{Barreiras} & \\multicolumn{%d}{c}{Camaçari} \\\\\n"
              "\\cmidrule(lr){2-%d}\\cmidrule(lr){%d-%d}\n" % (nA, nA, nA + 1, nA + 2, 2 * nA + 1), 1)
open(f, "w", encoding="utf-8").write(s)

# ------------------------------------------------------------------ Regressões (efeito flypaper)
r_ = t.copy()
r_["lnG"] = np.log(r_["desp_total_real_pc"]); r_["lnGc"] = np.log(r_["desp_corrente_real_pc"])
r_["lnT"] = np.log(r_["rec_transf_incondicionais_real_pc"]); r_["lnY"] = np.log(r_["pib_nominal_real_pc"])
r_["fe_cam"] = (r_.municipio == "Camaçari").astype(float)


def modelo(dados, dep):
    d = dados.dropna(subset=[dep, "lnT", "lnY"]).sort_values(["municipio", "ano"])
    X = np.column_stack([np.ones(len(d)), d.fe_cam, d.lnT, d.lnY, d.dummy_pos2018])
    return mqo_hc1(d[dep], X, ["const", "fe_cam", "lnT", "lnY", "D2018"]), d


especs = []
base = r_[(r_.ano >= A0) & (r_.ano <= 2023)]
amp = r_[(r_.ano >= 2014) & (r_.ano <= 2023)]
especs.append(("(1)", f"{A0}--2023", "ln $G$") + modelo(base, "lnG"))
especs.append(("(2)", "2014--2023", "ln $G$") + modelo(amp, "lnG"))
especs.append(("(3)", "2014--2023", "ln $G^{c}$") + modelo(amp, "lnGc"))
# (4) primeiras diferenças, 2014-2023 (a diferença elimina o efeito fixo)
fd = amp.sort_values(["municipio", "ano"]).copy()
for c in ["lnG", "lnT", "lnY"]:
    fd["d" + c] = fd.groupby("municipio")[c].diff()
fd["dD2018"] = fd.groupby("municipio")["dummy_pos2018"].diff()
fd = fd.dropna(subset=["dlnG", "dlnT", "dlnY"])
Xfd = np.column_stack([np.ones(len(fd)), fd.dlnT, fd.dlnY, fd.dD2018])
rfd = mqo_hc1(fd.dlnG, Xfd, ["const", "lnT", "lnY", "D2018"])
especs.append(("(4)", "2015--2023", "$\\Delta$ ln $G$", rfd, fd))

# VIF entre ln T e ln Y (amostra base, dentro do município)
db = base.dropna(subset=["lnT", "lnY"])
rTY = np.corrcoef(db.lnT - db.groupby("municipio").lnT.transform("mean"),
                  db.lnY - db.groupby("municipio").lnY.transform("mean"))[0, 1]
vif = 1 / (1 - rTY ** 2)

linhas, saida = [], []
def linha_coef(nome, rot):
    a, b = [rot], [""]
    for _, _, _, r, _ in especs:
        if nome in r["b"]:
            a.append(br(r["b"][nome], 3) + estrelas(r["p"][nome])); b.append("(" + br(r["ep"][nome], 3) + ")")
        else:
            a.append(""); b.append("")
    return [a, b]
for nome, rot in [("lnT", "ln $T$ ($\\beta_T$)"), ("lnY", "ln $Y$ ($\\beta_Y$)"),
                  ("D2018", "$D2018$")]:
    linhas += linha_coef(nome, rot)
linhas.append("\\midrule")
linhas.append(["Efeito fixo de município"] + ["Sim", "Sim", "Sim", "Dif."])
dif_l, p_l, mg_l = ["$\\beta_T - \\beta_Y$"], ["$p$ ($H_0$: $\\beta_T = \\beta_Y$)"], ["Efeito marginal $T$ / $Y$$^{b}$"]
for nm, per, dep, r, d in especs:
    dlt, pv = wald_igualdade(r, "lnT", "lnY")
    dif_l.append(br(dlt, 3)); p_l.append(br(pv, 3))
    Gm = d["desp_corrente_real_pc"].mean() if "G^{c}" in dep else d["desp_total_real_pc"].mean()
    mT = r["b"]["lnT"] * Gm / d["rec_transf_incondicionais_real_pc"].mean()
    mY = r["b"]["lnY"] * Gm / d["pib_nominal_real_pc"].mean()
    mg_l.append(br(mT, 2) + " / " + br(mY, 3))
    saida.append({"modelo": nm, "periodo": per, "dependente": dep, "n": r["n"], "r2": r["r2"], "dw": r["dw"],
                  **{f"b_{k}": v for k, v in r["b"].items()}, **{f"ep_{k}": v for k, v in r["ep"].items()},
                  **{f"p_{k}": v for k, v in r["p"].items()}, "bT_menos_bY": dlt, "p_igualdade": pv,
                  "efeito_marginal_T": mT, "efeito_marginal_Y": mY})
linhas += [dif_l, p_l, mg_l]
linhas.append(["Observações"] + [str(r["n"]) for *_, r, _ in especs])
linhas.append(["$R^2$"] + [br(r["r2"], 3) for *_, r, _ in especs])
linhas.append(["Durbin-Watson"] + [br(r["dw"], 2) for *_, r, _ in especs])
pd.DataFrame(saida).to_csv(os.path.join(RES, "regressoes.csv"), index=False, encoding="utf-8-sig")
cab = ["", *[f"{nm}" for nm, *_ in especs]]
tabela_tex("tab_regressao.tex", "Gasto municipal, transferências incondicionais e renda local: estimativas por MQO",
           "tab:regressao", cab,
           [[f"Período"] + [per for _, per, *_ in especs], [f"Variável dependente"] + [dep for _, _, dep, *_ in especs],
            "\\midrule"] + linhas,
           ("Nota: erros-padrão robustos à heterocedasticidade (HC1) entre parênteses; *** $p<0{,}01$, ** $p<0{,}05$, * $p<0{,}1$. "
            "$G$ = despesa total, $G^{c}$ = despesa corrente, $T$ = transferências incondicionais, $Y$ = PIB; valores reais per capita (preços de %d), em logaritmo. Modelos (1)--(3): efeito fixo de município "
            "(constante e dummy de Camaçari omitidas). Modelo (4): primeiras diferenças, com constante. A amostra termina em 2023 "
            "porque o PIB municipal não está disponível para 2024--2025. Correlação entre ln $T$ e ln $Y$ dentro do município na "
            "amostra (1): %s (VIF = %s). (b) Efeito marginal implícito = coeficiente $\\times$ média de $G$ / média da variável: "
            "reais de gasto associados a R\\$ 1 adicional de transferência e de PIB. Com poucas observações, os resultados são "
            "exploratórios e indicam associação, não causalidade. Fonte: elaboração própria.") % (ANO_BASE, br(rTY, 2), br(vif, 1)),
           alinhamento="lcccc")

# ------------------------------------------------------------------ Tabela: Hungria
ANOS_H = [a for a in [2012, 2013, 2016, 2019, 2022, 2025] if a in h.index]
IH = [("loc_despesa_pct_pib", "Despesa dos governos locais (\\% do PIB)", lambda x: br(x, 1)),
      ("loc_receita_pct_pib", "Receita dos governos locais (\\% do PIB)", lambda x: br(x, 1)),
      ("loc_despesa_sobre_despesa_gg", "Despesa local / desp. do gov. geral", pct),
      ("loc_transf_correntes_sobre_receita", "Transf. correntes / receita local", pct),
      ("loc_transf_total_sobre_receita", "Transf. correntes e de capital / receita", pct),
      ("loc_impostos_sobre_receita", "Impostos locais / receita local", pct),
      ("loc_despesa_huf_real_pc", "Despesa local per capita (mil HUF de %d)" % ANO_BASE, lambda x: br(x / 1000, 0))]
linhas = [[rot] + [fmt(h.loc[a, v]) for a in ANOS_H] for v, rot, fmt in IH]
h[[v for v, _, _ in IH]].to_csv(os.path.join(RES, "indicadores_hungria.csv"), encoding="utf-8-sig")
tabela_tex("tab_hungria.tex", "Hungria: tamanho e financiamento dos governos locais (setor S1313)", "tab:hungria",
           ["Indicador"] + [str(a) for a in ANOS_H], linhas,
           ("Nota: governos locais = setor S1313 das contas nacionais (SEC 2010). Transferências correntes = D73 (dentro do governo "
            "geral); de capital = D9; impostos = D2 + D5 + D91. Valores reais deflacionados pelo HICP médio anual (preços de %d) e "
            "divididos pela população (Banco Mundial). Fonte: elaboração própria com dados do Eurostat (gov\\_10a\\_main, "
            "prc\\_hicp\\_aind) e do Banco Mundial.") % ANO_BASE, alinhamento="l" + "r" * len(ANOS_H))

# ------------------------------------------------------------------ Tabela: o que mudou em relação à P1
c = pd.read_csv(os.path.join("data", "registro_alteracoes", "comparacao_P1_P2.csv"))
ch = pd.read_csv(os.path.join("data", "registro_alteracoes", "hungria_P1_P2.csv"))
alt = c[c.tipo != "IGUAL"]
def faixa(sub):
    anos = sorted(sub.ano.unique()); return f"{anos[0]}--{anos[-1]}" if len(anos) > 1 and anos[-1] - anos[0] == len(anos) - 1 else ", ".join(map(str, anos))
L = []
g = alt[alt.variavel_P1 == "Receita_Total"]
if len(g): L.append(["Receita total (Camaçari)", faixa(g), "P1 excluiu as receitas intra-orçamentárias, mas não as despesas",
                     f"+{br(100 * g.dif_rel.min(), 1)} a +{br(100 * g.dif_rel.max(), 1)}\\%"])
g = alt[alt.variavel_P1 == "FPM"]
if len(g): L.append(["FPM", ", ".join(f"{m[:3]}. {a}" for m, a in zip(g.municipio, g.ano)), "P1 omitiu as parcelas de 1\\%",
                     f"+{br(100 * g.dif_rel.min(), 1)} a +{br(100 * g.dif_rel.max(), 1)}\\%"])
g = alt[alt.variavel_P1 == "Funcao_Saude"]
if len(g): L.append(["Despesa em saúde (Camaçari)", faixa(g), "P1 gravou zero; valor da DCA: R\\$ %s mi" % br(g.valor_P2.iloc[0] / 1e6, 1), "--"])
g = alt[(alt.variavel_P1 == "População") & (alt.tipo == "CORRIGIDO")]
if len(g): L.append(["População (Barreiras)", faixa(g), "Dígitos trocados na P1 (%s; IBGE: %s)" % (br(g.valor_P1.iloc[0]), br(g.valor_P2.iloc[0])),
                     f"{br(100 * g.dif_rel.iloc[0], 1)}\\%"])
g = alt[alt.tipo == "PREENCHIDO"]
if len(g): L.append(["População", "2022--2023", "Ausente na P1; Censo 2022 e interpolação", "--"])
rr = t[t.ano.between(2014, 2023)]
vira = rr[(rr.resultado_bruto_sem_deducao > 0) & (rr.resultado_orcamentario < 0)]
L.append(["Resultado orçamentário", "2014--2023", "P1 não descontou a dedução do FUNDEB da receita (R\\$ %s a %s mi/ano); "
          "superávit vira déficit em %d anos (%s)" % (br(rr.ded_fundeb_total.min() / 1e6, 0), br(rr.ded_fundeb_total.max() / 1e6, 0),
          len(vira), ", ".join(f"{m[:3]}. {a}" for m, a in zip(vira.municipio, vira.ano))), "--"])
L.append(["Despesa por função", "vários", "Zeros da P1 tratados como ausentes", "--"])
L.append(["Hungria: receita e despesa do gov. geral", "2014--2021", "P1 estimou em HUF; P2 usa valor oficial do Eurostat",
          f"{br(100 * ch.dif_rel.min(), 1)} a +{br(100 * ch.dif_rel.max(), 1)}\\%"])
L.append(["Ampliações", "2014--2025", "2024--2025; ITR, IPVA, IPI, SUS, deduções do FUNDEB; governos locais húngaros", "--"])
iguais = (c.tipo == "IGUAL").sum(); total = len(c)
tabela_tex("tab_alteracoes.tex", "Principais alterações em relação ao trabalho anterior (P1)", "tab:alteracoes",
           ["Variável", "Anos", "Alteração", "Diferença"], L,
           (f"Nota: dos {total} valores municipais comparáveis (17 variáveis, 2 municípios, 2014--2023), {iguais} coincidem com a P1 "
            "(diferença menor que 0,1\\%). Diferença = (P2 $-$ P1)/P1. Registro completo em data/registro\\_alteracoes/. "
            "Fonte: elaboração própria."), alinhamento="p{3.3cm}p{2.1cm}p{6.2cm}r")

# ------------------------------------------------------------------ Figuras
def rotulo_final(ax, x, y, txt, cor):
    """Marca o ponto final e guarda o rótulo; posiciona_rotulos() desenha sem sobreposição."""
    ax.plot([x], [y], "o", ms=4, color=cor)
    if not hasattr(ax, "_rotulos"):
        ax._rotulos = []
    ax._rotulos.append([x, y, txt])


def posiciona_rotulos(ax, folga=0.075):
    """Rótulos diretos no fim das linhas, afastados verticalmente (mín. `folga` da altura do eixo)."""
    r = sorted(getattr(ax, "_rotulos", []), key=lambda z: z[1])
    y0, y1 = ax.get_ylim(); gap = folga * (y1 - y0)
    pos = [z[1] for z in r]
    for i in range(1, len(pos)):
        pos[i] = max(pos[i], pos[i - 1] + gap)
    excesso = pos[-1] - (y1 - gap / 2) if pos else 0
    if excesso > 0:
        pos = [p - excesso for p in pos]
        for i in range(len(pos) - 2, -1, -1):
            pos[i] = min(pos[i], pos[i + 1] - gap)
    for (x, y, txt), yp in zip(r, pos):
        ax.annotate(txt, (x, y), xytext=(x + 0.35, yp), textcoords="data", va="center", fontsize=8, color=TINTA,
                    arrowprops=dict(arrowstyle="-", color=TINTA2, lw=0.5) if abs(yp - y) > gap * 0.3 else None)

# F1: FPM, cota de ICMS e impostos próprios per capita (small multiples, mesma escala)
FON = [("rec_fpm_real_pc", "FPM", "#2a78d6"), ("rec_icms_cota_real_pc", "Cota ICMS", "#eb6834"),
       ("impostos_proprios_real_pc", "Impostos próprios", "#1baf7a")]
fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.0), sharey=True)
for ax, m in zip(axs, MUN):
    d = j[j.municipio == m]
    for v, rot, cor in FON:
        ax.plot(d.ano, d[v], color=cor, lw=2, marker="o", ms=3)
        rotulo_final(ax, d.ano.iloc[-1], d[v].iloc[-1], rot, cor)
    ax.set_title(m, fontsize=10, color=TINTA, loc="left")
    ax.set_xlim(A0 - 0.3, A1 + 2.6); ax.set_xticks(range(A0, A1 + 1, 3))
for ax in axs:
    posiciona_rotulos(ax)
axs[0].set_ylabel(f"R$ de {ANO_BASE} per capita")
axs[0].yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: br(x)))
fig.tight_layout(); fig.savefig(os.path.join(FIG, "fig_fontes_receita.pdf")); fig.savefig(os.path.join(FIG, "fig_fontes_receita.png"), dpi=200)
plt.close(fig)

# F2: Hungria - tamanho (% do PIB) e financiamento (% da receita local), dois painéis (um eixo cada)
hh = h.loc[h.index >= 2012]
fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.0))
ax = axs[0]
for v, rot, cor in [("loc_despesa_pct_pib", "Despesa", "#2a78d6"), ("loc_receita_pct_pib", "Receita", "#eb6834")]:
    ax.plot(hh.index, hh[v], color=cor, lw=2, marker="o", ms=3); rotulo_final(ax, hh.index[-1], hh[v].iloc[-1], rot, cor)
ax.set_title("(a) Governos locais, % do PIB", fontsize=10, loc="left", color=TINTA); ax.set_ylim(0, None)
ax = axs[1]
for v, rot, cor in [("loc_transf_correntes_sobre_receita", "Transf. correntes", "#2a78d6"),
                    ("loc_impostos_sobre_receita", "Impostos locais", "#1baf7a")]:
    ax.plot(hh.index, 100 * hh[v], color=cor, lw=2, marker="o", ms=3); rotulo_final(ax, hh.index[-1], 100 * hh[v].iloc[-1], rot, cor)
ax.set_title("(b) % da receita local", fontsize=10, loc="left", color=TINTA); ax.set_ylim(0, None)
for ax in axs:
    ax.set_xlim(2011.6, 2025 + 3.4); ax.set_xticks(range(2012, 2026, 3)); ax.axvspan(2012.5, 2013.5, color=GRADE, alpha=0.6, lw=0)
    ax.text(2013, ax.get_ylim()[1] * 0.02, "2013", ha="center", va="bottom", fontsize=7, color=TINTA2)
    posiciona_rotulos(ax, folga=0.08)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "fig_hungria_local.pdf")); fig.savefig(os.path.join(FIG, "fig_hungria_local.png"), dpi=200)
plt.close(fig)

# F3: dependência de transferências - municípios × governos locais húngaros (mesmo eixo, %)
fig, ax = plt.subplots(figsize=(7.2, 3.2))
for m in MUN:
    d = t[(t.ano >= 2014)].query("municipio == @m")
    ax.plot(d.ano, 100 * d.transf_corr_sobre_rc_sem_extra, color=COR[m], lw=2, marker="o", ms=3)
    rotulo_final(ax, d.ano.iloc[-1], 100 * d.transf_corr_sobre_rc_sem_extra.iloc[-1], m, COR[m])
hh2 = h.loc[h.index >= 2014]
ax.plot(hh2.index, 100 * hh2.loc_transf_correntes_sobre_receita, color=COR["Hungria"], lw=2, marker="s", ms=3)
rotulo_final(ax, hh2.index[-1], 100 * hh2.loc_transf_correntes_sobre_receita.iloc[-1], "Hungria (gov. locais)", COR["Hungria"])
ax.set_ylabel("Transferências correntes / receita (%)"); ax.set_ylim(0, 100)
ax.set_xlim(2013.6, 2025 + 2.4); ax.set_xticks(range(2014, 2026, 2))
posiciona_rotulos(ax)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "fig_dependencia.pdf")); fig.savefig(os.path.join(FIG, "fig_dependencia.png"), dpi=200)
plt.close(fig)

# ------------------------------------------------------------------ resumo no console
print(pd.DataFrame(saida)[["modelo", "periodo", "n", "b_lnT", "p_lnT", "b_lnY", "p_lnY", "b_D2018", "p_igualdade",
                           "efeito_marginal_T", "efeito_marginal_Y", "r2", "dw"]].round(3).to_string(index=False))
print(f"corr(lnT, lnY) dentro = {rTY:.2f}; VIF = {vif:.1f}")
print(pd.DataFrame(comp).pivot_table(index="indicador", columns=["municipio", "ano"], values="valor").round(3).to_string())
print("Arquivos:", sorted(os.listdir(TAB)), sorted(os.listdir(FIG)))


In [ ]:
%%writefile codigo/gera_dicionario.py
"""
Gera o dicionário de variáveis (docs/dicionario_variaveis.csv e .md) a partir das definições abaixo
e da cobertura observada nos dados tratados (anos com valor por unidade). Rodar depois do tratamento.
Uso: python codigo/gera_dicionario.py
"""
import os
import pandas as pd

SIC = "Siconfi/DCA - https://apidatalake.tesouro.gov.br/ords/siconfi/tt/dca"
SID = "IBGE/SIDRA - https://sidra.ibge.gov.br/tabela/{}"
EUR = "Eurostat - https://ec.europa.eu/eurostat/databrowser/view/{}/default/table"
WB = "Banco Mundial (WDI) - https://data.worldbank.org/indicator/{}"
AUS = "Ausente permanece vazio (NaN); nunca substituído por zero"
RB = "DCA Anexo I-C, coluna 'Receitas Brutas Realizadas'"
DE = "DCA, coluna 'Despesas Empenhadas'"

# nome: (definição, fonte, unidade, transformação/fórmula)
MUN = {
    "rec_total": ("Receita orçamentária total, incluindo intra-orçamentárias (linha 'TotalReceitas')", SIC, "R$ correntes", RB),
    "rec_corrente": ("Receitas correntes (natureza 1.0.0.0)", SIC, "R$ correntes", RB),
    "rec_tributaria": ("Impostos, taxas e contribuições de melhoria (natureza 1.1)", SIC, "R$ correntes", RB),
    "rec_transf_correntes": ("Transferências correntes (natureza 1.7)", SIC, "R$ correntes", RB),
    "rec_transf_uniao": ("Transferências correntes da União (1.7.2.1 até 2017; 1.7.1 a partir de 2018)", SIC, "R$ correntes", RB + "; mapeamento dependente do ano (quebra de 2018)"),
    "rec_transf_estados": ("Transferências correntes do Estado (1.7.2.2 até 2017; 1.7.2 a partir de 2018)", SIC, "R$ correntes", RB + "; mapeamento dependente do ano"),
    "rec_iptu": ("IPTU", SIC, "R$ correntes", RB + "; conta identificada por código + rótulo (rubricas_receita.py); a partir de 2018 inclui multas e dívida ativa do imposto"),
    "rec_itbi": ("ITBI", SIC, "R$ correntes", RB + "; código + rótulo"),
    "rec_iss": ("ISS/ISSQN", SIC, "R$ correntes", RB + "; código + rótulo"),
    "rec_fpm": ("Cota-parte do FPM, incluindo as parcelas adicionais de 1% (julho, setembro, dezembro)", SIC, "R$ correntes", RB + "; soma das contas de nível mais alto que casam com o rótulo"),
    "rec_itr": ("Cota-parte do ITR", SIC, "R$ correntes", RB),
    "rec_icms_cota": ("Cota-parte do ICMS (exclui a desoneração da LC 87/96)", SIC, "R$ correntes", RB),
    "rec_ipva_cota": ("Cota-parte do IPVA", SIC, "R$ correntes", RB),
    "rec_ipi_exp_cota": ("Cota-parte do IPI-exportação", SIC, "R$ correntes", RB),
    "rec_iof_ouro": ("Cota-parte do IOF-ouro", SIC, "R$ correntes", RB + "; sem lançamento nos dois municípios (vazio)"),
    "rec_fundeb": ("Transferências do FUNDEB, incluindo a complementação da União (bruto, antes da dedução)", SIC, "R$ correntes", RB),
    "rec_outras_multigov": ("Outras transferências multigovernamentais (extraordinárias; Barreiras 2017)", SIC, "R$ correntes", RB),
    "rec_sus": ("Transferências do SUS (União e Estado)", SIC, "R$ correntes", RB),
    "rec_transf_incondicionais": ("Transferências incondicionais: FPM + ITR + cotas de ICMS, IPVA e IPI-exportação + IOF-ouro", SIC, "R$ correntes", "Soma; vazio só se todos os componentes faltarem"),
    "ded_fundeb_corrente": ("Dedução para formação do FUNDEB sobre as receitas correntes", SIC, "R$ correntes", "DCA Anexo I-C, coluna 'Deduções - FUNDEB'"),
    "ded_fundeb_total": ("Dedução para o FUNDEB sobre a receita total", SIC, "R$ correntes", "DCA Anexo I-C, coluna 'Deduções - FUNDEB'"),
    "desp_total": ("Despesa total empenhada (Total Geral da Despesa, com intra-orçamentárias)", SIC, "R$ correntes", "DCA Anexo I-D, " + DE),
    "desp_corrente": ("Despesas correntes (3.0)", SIC, "R$ correntes", "DCA Anexo I-D, " + DE),
    "desp_pessoal": ("Pessoal e encargos sociais (3.1)", SIC, "R$ correntes", "DCA Anexo I-D, " + DE),
    "desp_capital": ("Despesas de capital (4.0)", SIC, "R$ correntes", "DCA Anexo I-D, " + DE),
    "desp_educacao": ("Despesa na função 12 - Educação", SIC, "R$ correntes", "DCA Anexo I-E, " + DE),
    "desp_saude": ("Despesa na função 10 - Saúde", SIC, "R$ correntes", "DCA Anexo I-E, " + DE),
    "populacao": ("População residente", SID.format("6579") + "; Censo 2022: " + SID.format("4714"), "pessoas",
                  "Estimativas do IBGE; 2022 = Censo; 2023 = interpolação linear Censo 2022-estimativa 2024 (coluna pop_fonte)"),
    "pib_nominal": ("PIB municipal a preços correntes", SID.format("5938"), "R$ correntes", "Tabela 5938 (R$ mil) × 1.000; disponível até 2023"),
    "vab_agro": ("Valor adicionado bruto da agropecuária", SID.format("5938"), "R$ correntes", "× 1.000"),
    "vab_industria": ("Valor adicionado bruto da indústria", SID.format("5938"), "R$ correntes", "× 1.000"),
    "vab_servicos": ("Valor adicionado bruto dos serviços (exceto adm. pública)", SID.format("5938"), "R$ correntes", "× 1.000"),
    "vab_adm_publica": ("Valor adicionado bruto da administração pública", SID.format("5938"), "R$ correntes", "× 1.000"),
    "ipca_media_anual": ("IPCA, número-índice (dez/1993 = 100), média dos 12 meses do ano", SID.format("1737"), "índice", "Média aritmética mensal"),
    "fator_deflator": ("Fator para converter valores do ano t a preços de 2025", SID.format("1737"), "razão", "IPCA_médio(2025) / IPCA_médio(t)"),
    "impostos_proprios": ("IPTU + ISS + ITBI", SIC, "R$ correntes", "Soma; vazio se algum componente faltar"),
    "transf_sobre_rec_corrente": ("Transferências correntes / receita corrente", SIC, "razão", "rec_transf_correntes / rec_corrente (nominais do mesmo ano)"),
    "tributaria_sobre_rec_corrente": ("Receita tributária / receita corrente", SIC, "razão", "rec_tributaria / rec_corrente"),
    "transf_incond_sobre_rec_corrente": ("Transferências incondicionais / receita corrente", SIC, "razão", "rec_transf_incondicionais / rec_corrente"),
    "rec_corrente_sem_extraord": ("Receita corrente sem as transferências extraordinárias", SIC, "R$ correntes", "rec_corrente - rec_outras_multigov"),
    "rec_total_liq_fundeb": ("Receita total líquida da dedução do FUNDEB", SIC, "R$ correntes", "rec_total - ded_fundeb_total"),
    "resultado_orcamentario": ("Resultado orçamentário", SIC, "R$ correntes", "rec_total_liq_fundeb - desp_total"),
    "resultado_bruto_sem_deducao": ("Resultado sem descontar o FUNDEB (definição da P1, para comparação)", SIC, "R$ correntes", "rec_total - desp_total"),
    "dummy_pos2018": ("1 a partir de 2018 (nova classificação da receita)", "construída", "0/1", "ano >= 2018"),
}
HUN = {
    "loc_receita_mio_huf": ("Receita total dos governos locais (S1313, TR)", EUR.format("gov_10a_main"), "milhões de HUF correntes", "Valor oficial"),
    "loc_despesa_mio_huf": ("Despesa total dos governos locais (S1313, TE)", EUR.format("gov_10a_main"), "milhões de HUF correntes", "Valor oficial"),
    "loc_receita_pct_pib": ("Receita dos governos locais em % do PIB", EUR.format("gov_10a_main"), "% do PIB", "Unidade PC_GDP do Eurostat"),
    "loc_despesa_pct_pib": ("Despesa dos governos locais em % do PIB", EUR.format("gov_10a_main"), "% do PIB", "Unidade PC_GDP"),
    "loc_transf_correntes_mio_huf": ("Transferências correntes recebidas de dentro do governo geral (D73REC)", EUR.format("gov_10a_main"), "milhões de HUF", "Valor oficial"),
    "loc_transf_capital_mio_huf": ("Transferências de capital recebidas (D9REC; inclui fundos da UE)", EUR.format("gov_10a_main"), "milhões de HUF", "Valor oficial"),
    "loc_impostos_mio_huf": ("Impostos dos governos locais (D2REC + D5REC + D91REC)", EUR.format("gov_10a_main"), "milhões de HUF", "Soma"),
    "loc_transf_correntes_sobre_receita": ("Transferências correntes / receita local", EUR.format("gov_10a_main"), "razão", "D73REC / TR"),
    "loc_transf_total_sobre_receita": ("Transferências correntes e de capital / receita local", EUR.format("gov_10a_main"), "razão", "(D73REC + D9REC) / TR"),
    "loc_impostos_sobre_receita": ("Impostos locais / receita local", EUR.format("gov_10a_main"), "razão", "(D2 + D5 + D91) / TR"),
    "loc_despesa_sobre_despesa_gg": ("Despesa local / despesa do governo geral (grau de descentralização)", EUR.format("gov_10a_main"), "razão", "TE(S1313) / TE(S13)"),
    "gg_receita_mio_huf": ("Receita total do governo geral (S13)", EUR.format("gov_10a_main"), "milhões de HUF", "Valor oficial"),
    "gg_despesa_mio_huf": ("Despesa total do governo geral (S13)", EUR.format("gov_10a_main"), "milhões de HUF", "Valor oficial"),
    "hicp_2015_100": ("Índice harmonizado de preços ao consumidor, média anual", EUR.format("prc_hicp_aind"), "índice (2015 = 100)", "Unidade INX_A_AVG"),
    "fator_deflator": ("Fator para preços de 2025 (Hungria)", EUR.format("prc_hicp_aind"), "razão", "HICP(2025) / HICP(t)"),
    "populacao": ("População total da Hungria", WB.format("SP.POP.TOTL"), "pessoas", "Valor original"),
    "loc_despesa_huf_real_pc": ("Despesa local real per capita", "Eurostat + Banco Mundial", "HUF de 2025 por habitante", "loc_despesa_mio_huf × 10^6 × fator_deflator / populacao"),
}


def cobertura(df, col, por):
    if col not in df.columns:
        return "não gerada"
    partes = []
    for u, g in df.groupby(por) if por else [("Hungria", df)]:
        anos = g.loc[g[col].notna(), "ano"].astype(int)
        partes.append(f"{u}: {anos.min()}-{anos.max()} ({len(anos)} anos)" if len(anos) else f"{u}: sem dados")
    return "; ".join(partes)


def main():
    tm = pd.read_csv(os.path.join("data", "tratados", "tratado_municipios.csv"))
    th = pd.read_csv(os.path.join("data", "tratados", "tratado_hungria.csv"))
    linhas = []
    for nome, (dfn, fonte, un, tr) in MUN.items():
        linhas.append({"base": "tratado_municipios.csv", "variavel": nome, "definicao": dfn, "fonte": fonte, "unidade": un,
                       "cobertura": cobertura(tm, nome, "municipio"), "ausentes": AUS, "transformacao": tr})
    linhas.append({"base": "tratado_municipios.csv", "variavel": "<var>_real", "definicao": "Valor real da variável monetária",
                   "fonte": "derivada", "unidade": "R$ de 2025", "cobertura": "igual à da variável original", "ausentes": AUS,
                   "transformacao": "<var> × fator_deflator"})
    linhas.append({"base": "tratado_municipios.csv", "variavel": "<var>_real_pc", "definicao": "Valor real per capita",
                   "fonte": "derivada", "unidade": "R$ de 2025 por habitante", "cobertura": "igual à da variável original",
                   "ausentes": AUS, "transformacao": "<var>_real / populacao"})
    for nome, (dfn, fonte, un, tr) in HUN.items():
        linhas.append({"base": "tratado_hungria.csv", "variavel": nome, "definicao": dfn, "fonte": fonte, "unidade": un,
                       "cobertura": cobertura(th, nome, None), "ausentes": AUS, "transformacao": tr})
    d = pd.DataFrame(linhas)
    os.makedirs("docs", exist_ok=True)
    d.to_csv(os.path.join("docs", "dicionario_variaveis.csv"), index=False, encoding="utf-8-sig")
    with open(os.path.join("docs", "dicionario_variaveis.md"), "w", encoding="utf-8") as f:
        f.write("# Dicionário de variáveis\n\nGerado por `codigo/gera_dicionario.py`. Valores ausentes: " + AUS +
                ".\nPreços constantes de 2025 (Brasil: IPCA médio anual; Hungria: HICP médio anual).\n\n")
        for base, g in d.groupby("base", sort=False):
            f.write(f"## {base}\n\n| Variável | Definição | Fonte | Unidade | Cobertura | Transformação |\n|---|---|---|---|---|---|\n")
            for _, r in g.iterrows():
                f.write(f"| `{r.variavel}` | {r.definicao} | {r.fonte} | {r.unidade} | {r.cobertura} | {r.transformacao} |\n")
            f.write("\n")
    print(len(d), "variáveis documentadas")


if __name__ == "__main__":
    main()


## 1. Coleta – municípios

In [ ]:
!python codigo/coleta_municipios.py

## 2. Coleta – Hungria e Brasil

In [ ]:
!python codigo/coleta_hungria.py

## 3. Tratamento

In [ ]:
!python codigo/tratamento.py
!python codigo/tratamento_hungria.py

## 4. Base da P1 e auditoria

In [ ]:
!rm -rf data/original/P1_anterior
!git clone -q --depth 1 https://github.com/Jhons777/P1-Finan-as-do-Setor-Publico---Jo-o-Paulo-Cortes-Da-Cunha-Dutra-241021838 data/original/P1_anterior
!rm -rf data/original/P1_anterior/.git
!python codigo/importa_base_original.py data/original/P1_anterior

## 5. Registro de alterações, análise e dicionário

In [ ]:
!python codigo/compara_original_atualizado.py
!python codigo/analise.py
!python codigo/gera_dicionario.py

## 6. Baixar tudo

In [ ]:
!zip -qr P2_completo.zip data codigo artigo resultados docs
from google.colab import files
files.download('P2_completo.zip')